<a href="https://colab.research.google.com/github/gear66me-ui/IERS_TN36_V01_MASTER-1/blob/main/IERS_TN36_V01_MASTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Cell

In [ ]:
# V0001
# IERS_TN36_V01_MASTER - STARTUP CELL

!pip install -q astroquery astropy

import os
import sys
import math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from astropy.time import Time
from astropy.coordinates import EarthLocation
import astropy.units as u

from astroquery.jplhorizons import Horizons

from google.colab import drive
drive.mount("/content/drive")

print("V0001 STARTUP COMPLETE")

# V0001

Mounted at /content/drive
V0001 STARTUP COMPLETE


In [ ]:
# IERS-0002
# Audit reference: formatted IERS TN36 North Pole–Miami Halley parallax report from V02-0029.

import os
import sys
import math
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd

from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar, brentq

from astroquery.jplhorizons import Horizons

import astropy.units as u
from astropy.time import Time
from astropy.coordinates import EarthLocation
from astropy.utils import iers

VERSION = "IERS-0002"

AU_KM = 149597870.7
ARCSEC_PER_RAD = 206264.80624709636
SUN_RADIUS_KM = 695700.0
VENUS_RADIUS_KM = 6051.8
EARTH_RADIUS_KM = 6378.137
IAU_SOLAR_PARALLAX_ARCSEC = 8.794148
LOCAL_TZ = timezone(timedelta(hours=-5))

START = "2012-Jun-05 20:00"
STOP = "2012-Jun-06 07:30"
STEP = "1m"

PROJECT_ROOT = "/content/IERS_TN36_OUTPUT"
OUTPUT_CSV_DIR = os.path.join(PROJECT_ROOT, "OUTPUT_CSV")
os.makedirs(OUTPUT_CSV_DIR, exist_ok=True)

OUT_CSV = os.path.join(
    OUTPUT_CSV_DIR,
    f"{VERSION}_NORTH_POLE_MIAMI_FORMATTED_ENGINEERING_REPORT.csv"
)

SITE_A = {
    "key": "NORTH_POLE",
    "label": "North Pole",
    "lon_deg_east": 0.0,
    "lat_deg": 90.0,
    "height_m": 0.0,
}

SITE_B = {
    "key": "MIAMI_FL",
    "label": "Miami FL",
    "lon_deg_east": -80.1918,
    "lat_deg": 25.7617,
    "height_m": 0.0,
}

iers.conf.auto_download = True
iers.conf.iers_degraded_accuracy = "warn"


def norm(v):
    return float(np.sqrt(np.dot(v, v)))


def unit(v):
    n = norm(v)
    if n == 0.0:
        raise RuntimeError("Zero vector.")
    return np.asarray(v, dtype=float) / n


def angular_sep_arcsec(a, b):
    c = float(np.clip(np.dot(unit(a), unit(b)), -1.0, 1.0))
    return math.acos(c) * ARCSEC_PER_RAD


def horizons_geocenter_vectors(target_id, prefix):
    obj = Horizons(
        id=target_id,
        location="500@399",
        epochs={"start": START, "stop": STOP, "step": STEP},
    )
    vec = obj.vectors().to_pandas()
    out = pd.DataFrame()
    out["jd_tdb"] = vec["datetime_jd"].astype(float)
    out["utc"] = vec["datetime_str"].astype(str)
    out[f"{prefix}_x_km"] = vec["x"].astype(float) * AU_KM
    out[f"{prefix}_y_km"] = vec["y"].astype(float) * AU_KM
    out[f"{prefix}_z_km"] = vec["z"].astype(float) * AU_KM
    return out


def build_master_geocenter():
    sun = horizons_geocenter_vectors("10", "GEOCENTER_SUN")
    venus = horizons_geocenter_vectors("299", "GEOCENTER_VENUS")
    return sun.merge(venus, on=["jd_tdb", "utc"], how="inner")


def build_cache(df):
    cache = {
        "jd_tdb": df["jd_tdb"].to_numpy(dtype=float),
        "utc": df["utc"].astype(str).tolist(),
    }
    for col in df.columns:
        if col.endswith("_km"):
            cache[col] = CubicSpline(
                cache["jd_tdb"],
                df[col].to_numpy(dtype=float),
                bc_type="natural",
            )
    return cache


def vec_at(cache, prefix, jd_tdb):
    return np.array(
        [
            float(cache[f"{prefix}_x_km"](jd_tdb)),
            float(cache[f"{prefix}_y_km"](jd_tdb)),
            float(cache[f"{prefix}_z_km"](jd_tdb)),
        ],
        dtype=float,
    )


def utc_at(cache, jd_tdb):
    try:
        return Time(jd_tdb, format="jd", scale="tdb").utc.iso.replace(" ", " ")
    except Exception:
        jds = cache["jd_tdb"]
        idx = int(np.argmin(np.abs(jds - jd_tdb)))
        return cache["utc"][idx]


def earth_location(site):
    return EarthLocation.from_geodetic(
        lon=site["lon_deg_east"] * u.deg,
        lat=site["lat_deg"] * u.deg,
        height=site["height_m"] * u.m,
    )


def observer_gcrs_km(site, jd_tdb):
    t = Time(jd_tdb, format="jd", scale="tdb")
    loc = earth_location(site)
    pos, _vel = loc.get_gcrs_posvel(t)
    return np.array(
        [
            pos.x.to_value(u.km),
            pos.y.to_value(u.km),
            pos.z.to_value(u.km),
        ],
        dtype=float,
    )


def topocentric_body_vector(cache, site, body_prefix, jd_tdb):
    return vec_at(cache, body_prefix, jd_tdb) - observer_gcrs_km(site, jd_tdb)


def site_sep_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    return angular_sep_arcsec(sun, venus)


def geocenter_sep_arcsec(cache, jd_tdb):
    return angular_sep_arcsec(
        vec_at(cache, "GEOCENTER_SUN", jd_tdb),
        vec_at(cache, "GEOCENTER_VENUS", jd_tdb),
    )


def angular_radii_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    rs = math.atan2(SUN_RADIUS_KM, norm(sun)) * ARCSEC_PER_RAD
    rv = math.atan2(VENUS_RADIUS_KM, norm(venus)) * ARCSEC_PER_RAD
    return rs, rv


def contact_function(cache, site, event, jd_tdb):
    sep = site_sep_arcsec(cache, site, jd_tdb)
    rs, rv = angular_radii_arcsec(cache, site, jd_tdb)
    threshold = rs + rv if event in ["C1", "C4"] else rs - rv
    return sep - threshold


def find_roots(cache, site, event):
    jds = cache["jd_tdb"]
    vals = np.array([contact_function(cache, site, event, jd) for jd in jds], dtype=float)
    roots = []
    for i in range(len(jds) - 1):
        if not np.isfinite(vals[i]) or not np.isfinite(vals[i + 1]):
            continue
        if vals[i] == 0.0:
            roots.append(float(jds[i]))
        elif vals[i] * vals[i + 1] < 0.0:
            roots.append(
                float(
                    brentq(
                        lambda x: contact_function(cache, site, event, x),
                        jds[i],
                        jds[i + 1],
                        xtol=1e-13,
                        rtol=1e-13,
                        maxiter=100,
                    )
                )
            )
    return roots


def contact_range(cache):
    roots = []
    for site in [SITE_A, SITE_B]:
        roots.extend(find_roots(cache, site, "C1"))
        roots.extend(find_roots(cache, site, "C4"))
    if not roots:
        raise RuntimeError("No C1/C4 contacts found.")
    return min(roots), max(roots)


def find_closest(cache):
    jds = cache["jd_tdb"]
    vals = [geocenter_sep_arcsec(cache, jd) for jd in jds]
    i = int(np.argmin(vals))
    a = jds[max(0, i - 3)]
    b = jds[min(len(jds) - 1, i + 3)]
    res = minimize_scalar(
        lambda jd: geocenter_sep_arcsec(cache, jd),
        bounds=(a, b),
        method="bounded",
        options={"xatol": 1e-13},
    )
    return float(res.x)


def fixed_geocenter_basis(cache, jd_tdb):
    sun = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    n = unit(sun)
    ref = np.array([0.0, 0.0, 1.0])
    if norm(np.cross(ref, n)) < 1e-12:
        ref = np.array([1.0, 0.0, 0.0])
    xhat = unit(np.cross(ref, n))
    yhat = unit(np.cross(n, xhat))
    return n, xhat, yhat


def ray_screen_point_arcsec(cache, site, jd_tdb, basis):
    n, xhat, yhat = basis
    obs = observer_gcrs_km(site, jd_tdb)
    sun_geo = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    venus_geo = vec_at(cache, "GEOCENTER_VENUS", jd_tdb)

    ray = venus_geo - obs
    screen_point = sun_geo
    denom = float(np.dot(ray, n))
    if abs(denom) < 1e-14:
        raise RuntimeError("Ray nearly parallel to solar screen.")

    tau = float(np.dot(screen_point - obs, n) / denom)
    hit = obs + tau * ray
    screen_vec = hit - sun_geo
    es = norm(sun_geo)

    x = math.atan2(float(np.dot(screen_vec, xhat)), es) * ARCSEC_PER_RAD
    y = math.atan2(float(np.dot(screen_vec, yhat)), es) * ARCSEC_PER_RAD
    return np.array([x, y], dtype=float)


def pca_direction(points):
    pts = np.asarray(points, dtype=float)
    mu = pts.mean(axis=0)
    centered = pts - mu
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    d = vt[0]
    if d[0] < 0:
        d = -d
    return mu, unit(d)


def line_intersection(mu, d, mid, normal):
    a = np.column_stack([d, -normal])
    b = mid - mu
    sol, *_ = np.linalg.lstsq(a, b, rcond=None)
    return mu + sol[0] * d


def fit_r2(points, degree):
    pts = np.asarray(points, dtype=float)
    x = pts[:, 0] - np.mean(pts[:, 0])
    y = pts[:, 1]
    coeff = np.polyfit(x, y, degree)
    yhat = np.polyval(coeff, x)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")


def distances_at(cache, jd_tdb):
    es = norm(vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    ev = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb))
    vs = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb) - vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    return es, ev, vs


def compute_iers_guideline_method(cache):
    ca_jd = find_closest(cache)
    c1, c4 = contact_range(cache)
    use_jds = cache["jd_tdb"][(cache["jd_tdb"] >= c1) & (cache["jd_tdb"] <= c4)]
    basis = fixed_geocenter_basis(cache, ca_jd)

    mus = {}
    dirs = {}
    track_rows = []

    for site in [SITE_A, SITE_B]:
        pts = np.array([ray_screen_point_arcsec(cache, site, jd, basis) for jd in use_jds], dtype=float)
        mu, d = pca_direction(pts)

        mus[site["key"]] = mu
        dirs[site["key"]] = d

        angle = math.degrees(math.atan2(d[1], d[0]))

        track_rows.append(
            {
                "observer": site["label"],
                "lat_deg": site["lat_deg"],
                "lon_deg": site["lon_deg_east"],
                "theta_deg": angle,
                "linear_r2": fit_r2(pts, 1),
                "quad_r2": fit_r2(pts, 2),
                "cubic_r2": fit_r2(pts, 3),
            }
        )

    tangent = unit(dirs[SITE_A["key"]] + dirs[SITE_B["key"]])
    if tangent[0] < 0:
        tangent = -tangent

    normal = np.array([-tangent[1], tangent[0]])
    theta_common_deg = math.degrees(math.atan2(tangent[1], tangent[0]))
    theta_mean_deg = 0.5 * (track_rows[0]["theta_deg"] + track_rows[1]["theta_deg"])

    mid = 0.5 * (mus[SITE_A["key"]] + mus[SITE_B["key"]])
    aprime = line_intersection(mus[SITE_A["key"]], dirs[SITE_A["key"]], mid, normal)
    bprime = line_intersection(mus[SITE_B["key"]], dirs[SITE_B["key"]], mid, normal)

    abp_vec = bprime - aprime
    abp_obs_arcsec = float(np.sqrt(np.sum(abp_vec * abp_vec)))
    rho_obs_arcsec = abs(float(np.dot(abp_vec, normal)))

    es, ev, vs = distances_at(cache, ca_jd)

    abp_obs_km = math.tan(abp_obs_arcsec / ARCSEC_PER_RAD) * es
    b_eff_obs_km = abp_obs_km * ev / vs

    pi_obs_arcsec = rho_obs_arcsec * (ev / vs) * (EARTH_RADIUS_KM / b_eff_obs_km)
    pi_iau_arcsec = pi_obs_arcsec * (es / AU_KM)

    delta_pi_arcsec = pi_iau_arcsec - IAU_SOLAR_PARALLAX_ARCSEC
    delta_pi_percent = 100.0 * delta_pi_arcsec / IAU_SOLAR_PARALLAX_ARCSEC

    parallax = {
        "closest_jd": ca_jd,
        "closest_utc": utc_at(cache, ca_jd),
        "A′B′₍OBS₎": abp_obs_arcsec,
        "A′B′₍OBS₎_km": abp_obs_km,
        "ρ₍OBS₎": rho_obs_arcsec,
        "Bₑff₍OBS₎": b_eff_obs_km,
        "π₍OBS₎": pi_obs_arcsec,
        "π₍IAU₎": pi_iau_arcsec,
        "Δπ": delta_pi_arcsec,
        "Δπ_%": delta_pi_percent,
        "D_ES_AU": es / AU_KM,
        "D_EV_D_VS": ev / vs,
        "θ̄": theta_mean_deg,
        "θ₍COMMON₎": theta_common_deg,
    }

    return parallax, track_rows


def fmt6(x):
    return f"{float(x):.6f}"


def fmt9(x):
    return f"{float(x):.9f}"


def fmt12(x):
    return f"{float(x):.12f}"


def section(title):
    line = "═" * 84
    print()
    print(line)
    print(title.center(84))
    print(line)


def table(headers, rows):
    widths = [len(str(h)) for h in headers]

    for row in rows:
        for i, val in enumerate(row):
            widths[i] = max(widths[i], len(str(val)))

    fmt = "  ".join(f"{{:<{w}}}" for w in widths)

    print("─" * 84)
    print(fmt.format(*headers))
    print("─" * 84)

    for row in rows:
        print(fmt.format(*row))

    print("─" * 84)


def save_csv(parallax, track_rows):
    rows = [
        ["A′B′₍OBS₎", parallax["A′B′₍OBS₎"], "arcsec"],
        ["A′B′₍OBS₎", parallax["A′B′₍OBS₎_km"], "km"],
        ["ρ₍OBS₎", parallax["ρ₍OBS₎"], "arcsec"],
        ["Bₑff₍OBS₎", parallax["Bₑff₍OBS₎"], "km"],
        ["π₍OBS₎", parallax["π₍OBS₎"], "arcsec"],
        ["π₍IAU₎", parallax["π₍IAU₎"], "arcsec"],
        ["Δπ", parallax["Δπ"], "arcsec"],
        ["Δπ", parallax["Δπ_%"], "%"],
        ["D_ES/AU", parallax["D_ES_AU"], ""],
        ["D_EV/D_VS", parallax["D_EV_D_VS"], ""],
        ["θ̄", parallax["θ̄"], "deg"],
        ["θ₍COMMON₎", parallax["θ₍COMMON₎"], "deg"],
    ]

    for tr in track_rows:
        rows.append([f"θ₍{tr['observer']}₎", tr["theta_deg"], "deg"])
        rows.append([f"R²_LINEAR₍{tr['observer']}₎", tr["linear_r2"], ""])
        rows.append([f"R²_CUBIC₍{tr['observer']}₎", tr["cubic_r2"], ""])

    df = pd.DataFrame(rows, columns=["Quantity", "Value", "Unit"])
    df.to_csv(OUT_CSV, index=False, float_format="%.12f")
    return OUT_CSV


def print_report(parallax, track_rows, csv_path):
    print(f"CODE OUTPUT: {VERSION}")

    section("IERS PARALLAX SOLUTION")

    table(
        ["Quantity", "Value", "Units"],
        [
            ["A′B′₍OBS₎", fmt6(parallax["A′B′₍OBS₎"]), "arcsec"],
            ["A′B′₍OBS₎", fmt6(parallax["A′B′₍OBS₎_km"]), "km"],
            ["ρ₍OBS₎", fmt6(parallax["ρ₍OBS₎"]), "arcsec"],
            ["Bₑff₍OBS₎", fmt6(parallax["Bₑff₍OBS₎"]), "km"],
            ["π₍OBS₎", fmt6(parallax["π₍OBS₎"]), "arcsec"],
            ["π₍IAU₎", fmt6(parallax["π₍IAU₎"]), "arcsec"],
            ["Δπ", fmt6(parallax["Δπ"]), "arcsec"],
            ["Δπ", fmt6(parallax["Δπ_%"]), "%"],
        ],
    )

    section("TRACK GEOMETRY")

    table(
        ["Observer", "θ", "Linear R²", "Quadratic R²", "Cubic R²"],
        [
            [
                tr["observer"],
                fmt6(tr["theta_deg"]),
                fmt9(tr["linear_r2"]),
                fmt9(tr["quad_r2"]),
                fmt9(tr["cubic_r2"]),
            ]
            for tr in track_rows
        ]
        + [
            ["θ̄", fmt6(parallax["θ̄"]), "", "", ""],
            ["θ₍COMMON₎", fmt6(parallax["θ₍COMMON₎"]), "", "", ""],
        ],
    )

    section("CLOSEST APPROACH")

    table(
        ["Quantity", "Value", "Units"],
        [
            ["UTC", parallax["closest_utc"], ""],
            ["JD_TDB", fmt12(parallax["closest_jd"]), "d"],
        ],
    )

    section("GEOMETRY RATIOS")

    table(
        ["Quantity", "Value", "Units"],
        [
            ["D_ES/AU", fmt6(parallax["D_ES_AU"]), ""],
            ["D_EV/D_VS", fmt6(parallax["D_EV_D_VS"]), ""],
        ],
    )

    section("FILES")

    table(
        ["Type", "Path"],
        [
            ["CSV", csv_path],
        ],
    )

    print()
    print(datetime.now(LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S %z"))
    print(f"END OF CODE OUTPUT: {VERSION}")


def main():
    df = build_master_geocenter()
    cache = build_cache(df)
    parallax, track_rows = compute_iers_guideline_method(cache)
    csv_path = save_csv(parallax, track_rows)
    print_report(parallax, track_rows, csv_path)


if __name__ == "__main__":
    main()
# IERS-0002

CODE OUTPUT: IERS-0002

════════════════════════════════════════════════════════════════════════════════════
                               IERS PARALLAX SOLUTION                               
════════════════════════════════════════════════════════════════════════════════════
────────────────────────────────────────────────────────────────────────────────────
Quantity   Value         Units 
────────────────────────────────────────────────────────────────────────────────────
A′B′₍OBS₎  14.342821     arcsec
A′B′₍OBS₎  10555.777283  km    
ρ₍OBS₎     14.342821     arcsec
Bₑff₍OBS₎  4197.408601   km    
π₍OBS₎     8.666390      arcsec
π₍IAU₎     8.794144      arcsec
Δπ         -0.000004     arcsec
Δπ         -0.000048     %     
────────────────────────────────────────────────────────────────────────────────────

════════════════════════════════════════════════════════════════════════════════════
                                   TRACK GEOMETRY                                   
═══════

In [ ]:
# %load IERS-0003_BLACK_SUBSCRIPT_ENGINEERING_REPORT.py
# IERS-0003
# Audit reference: formatted IERS North Pole Miami report derived from validated V02-0029 implementation.

import sys
import os
import math
import subprocess
from datetime import datetime, timezone, timedelta

VERSION = "IERS-0003"

AU_KM = 149597870.7
ARCSEC_PER_RAD = 206264.80624709636
SUN_RADIUS_KM = 695700.0
VENUS_RADIUS_KM = 6051.8
EARTH_RADIUS_KM = 6378.137
IAU_SOLAR_PARALLAX_ARCSEC = 8.794148
LOCAL_TZ = timezone(timedelta(hours=-5))

START = "2012-Jun-05 20:00"
STOP = "2012-Jun-06 07:30"
STEP = "1m"

SITE_A = {
    "key": "NORTH_POLE",
    "label": "North Pole",
    "lon_deg_east": 0.0,
    "lat_deg": 90.0,
    "height_m": 0.0,
}

SITE_B = {
    "key": "MIAMI_FL",
    "label": "Miami FL",
    "lon_deg_east": -80.1918,
    "lat_deg": 25.7617,
    "height_m": 0.0,
}

try:
    import numpy as np
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas"])
    import numpy as np
    import pandas as pd

try:
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "scipy"])
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq

try:
    from astroquery.jplhorizons import Horizons
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astroquery"])
    from astroquery.jplhorizons import Horizons

try:
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astropy"])
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers

# Use bundled/fetched IERS tables when available; allow degraded mode in Colab if network table access is unavailable.
iers.conf.auto_download = True
iers.conf.iers_degraded_accuracy = "warn"


def norm(v):
    return float(np.sqrt(np.dot(v, v)))


def unit(v):
    n = norm(v)
    if n == 0.0:
        raise RuntimeError("Zero vector.")
    return np.asarray(v, dtype=float) / n


def angular_sep_arcsec(a, b):
    c = float(np.clip(np.dot(unit(a), unit(b)), -1.0, 1.0))
    return math.acos(c) * ARCSEC_PER_RAD


def horizons_geocenter_vectors(target_id, prefix):
    obj = Horizons(
        id=target_id,
        location="500@399",
        epochs={"start": START, "stop": STOP, "step": STEP},
    )
    vec = obj.vectors().to_pandas()
    out = pd.DataFrame()
    out["jd_tdb"] = vec["datetime_jd"].astype(float)
    out["utc"] = vec["datetime_str"].astype(str)
    out[f"{prefix}_x_km"] = vec["x"].astype(float) * AU_KM
    out[f"{prefix}_y_km"] = vec["y"].astype(float) * AU_KM
    out[f"{prefix}_z_km"] = vec["z"].astype(float) * AU_KM
    return out


def build_master_geocenter():
    sun = horizons_geocenter_vectors("10", "GEOCENTER_SUN")
    venus = horizons_geocenter_vectors("299", "GEOCENTER_VENUS")
    return sun.merge(venus, on=["jd_tdb", "utc"], how="inner")


def build_cache(df):
    cache = {"jd_tdb": df["jd_tdb"].to_numpy(dtype=float), "utc": df["utc"].astype(str).tolist()}
    for col in df.columns:
        if col.endswith("_km"):
            cache[col] = CubicSpline(cache["jd_tdb"], df[col].to_numpy(dtype=float), bc_type="natural")
    return cache


def vec_at(cache, prefix, jd_tdb):
    return np.array([
        float(cache[f"{prefix}_x_km"](jd_tdb)),
        float(cache[f"{prefix}_y_km"](jd_tdb)),
        float(cache[f"{prefix}_z_km"](jd_tdb)),
    ], dtype=float)


def utc_at(cache, jd_tdb):
    try:
        return Time(jd_tdb, format="jd", scale="tdb").utc.iso.replace(" ", " ")
    except Exception:
        jds = cache["jd_tdb"]
        idx = int(np.argmin(np.abs(jds - jd_tdb)))
        return cache["utc"][idx]


def earth_location(site):
    # Astropy EarthLocation uses the ERFA/SOFA geodetic-to-geocentric implementation.
    # Geodetic longitude is east-positive, matching Horizons convention used in this project.
    return EarthLocation.from_geodetic(
        lon=site["lon_deg_east"] * u.deg,
        lat=site["lat_deg"] * u.deg,
        height=site["height_m"] * u.m,
    )


def observer_gcrs_km(site, jd_tdb):
    # IERS path: geodetic site -> ITRS -> GCRS, implemented through Astropy/ERFA using IERS EOP tables.
    # The time is supplied as TDB and internally converted by Astropy as required for Earth orientation.
    t = Time(jd_tdb, format="jd", scale="tdb")
    loc = earth_location(site)
    pos, _vel = loc.get_gcrs_posvel(t)
    return np.array([pos.x.to_value(u.km), pos.y.to_value(u.km), pos.z.to_value(u.km)], dtype=float)


def topocentric_body_vector(cache, site, body_prefix, jd_tdb):
    return vec_at(cache, body_prefix, jd_tdb) - observer_gcrs_km(site, jd_tdb)


def site_sep_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    return angular_sep_arcsec(sun, venus)


def geocenter_sep_arcsec(cache, jd_tdb):
    return angular_sep_arcsec(vec_at(cache, "GEOCENTER_SUN", jd_tdb), vec_at(cache, "GEOCENTER_VENUS", jd_tdb))


def angular_radii_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    rs = math.atan2(SUN_RADIUS_KM, norm(sun)) * ARCSEC_PER_RAD
    rv = math.atan2(VENUS_RADIUS_KM, norm(venus)) * ARCSEC_PER_RAD
    return rs, rv


def contact_function(cache, site, event, jd_tdb):
    sep = site_sep_arcsec(cache, site, jd_tdb)
    rs, rv = angular_radii_arcsec(cache, site, jd_tdb)
    threshold = rs + rv if event in ["C1", "C4"] else rs - rv
    return sep - threshold


def find_roots(cache, site, event):
    jds = cache["jd_tdb"]
    vals = np.array([contact_function(cache, site, event, jd) for jd in jds], dtype=float)
    roots = []
    for i in range(len(jds) - 1):
        if not np.isfinite(vals[i]) or not np.isfinite(vals[i + 1]):
            continue
        if vals[i] == 0.0:
            roots.append(float(jds[i]))
        elif vals[i] * vals[i + 1] < 0.0:
            roots.append(float(brentq(
                lambda x: contact_function(cache, site, event, x),
                jds[i],
                jds[i + 1],
                xtol=1e-13,
                rtol=1e-13,
                maxiter=100,
            )))
    return roots


def contact_range(cache):
    roots = []
    for site in [SITE_A, SITE_B]:
        roots.extend(find_roots(cache, site, "C1"))
        roots.extend(find_roots(cache, site, "C4"))
    if not roots:
        raise RuntimeError("No C1/C4 contacts found.")
    return min(roots), max(roots)


def find_closest(cache):
    jds = cache["jd_tdb"]
    vals = [geocenter_sep_arcsec(cache, jd) for jd in jds]
    i = int(np.argmin(vals))
    a = jds[max(0, i - 3)]
    b = jds[min(len(jds) - 1, i + 3)]
    res = minimize_scalar(
        lambda jd: geocenter_sep_arcsec(cache, jd),
        bounds=(a, b),
        method="bounded",
        options={"xatol": 1e-13},
    )
    return float(res.x)


def fixed_geocenter_basis(cache, jd_tdb):
    sun = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    n = unit(sun)
    ref = np.array([0.0, 0.0, 1.0])
    if norm(np.cross(ref, n)) < 1e-12:
        ref = np.array([1.0, 0.0, 0.0])
    xhat = unit(np.cross(ref, n))
    yhat = unit(np.cross(n, xhat))
    return n, xhat, yhat


def ray_screen_point_arcsec(cache, site, jd_tdb, basis):
    n, xhat, yhat = basis
    obs = observer_gcrs_km(site, jd_tdb)
    sun_geo = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    venus_geo = vec_at(cache, "GEOCENTER_VENUS", jd_tdb)
    ray = venus_geo - obs
    screen_point = sun_geo
    denom = float(np.dot(ray, n))
    if abs(denom) < 1e-14:
        raise RuntimeError("Ray nearly parallel to solar screen.")
    tau = float(np.dot(screen_point - obs, n) / denom)
    hit = obs + tau * ray
    screen_vec = hit - sun_geo
    es = norm(sun_geo)
    x = math.atan2(float(np.dot(screen_vec, xhat)), es) * ARCSEC_PER_RAD
    y = math.atan2(float(np.dot(screen_vec, yhat)), es) * ARCSEC_PER_RAD
    return np.array([x, y], dtype=float)


def pca_direction(points):
    pts = np.asarray(points, dtype=float)
    mu = pts.mean(axis=0)
    centered = pts - mu
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    d = vt[0]
    if d[0] < 0:
        d = -d
    return mu, unit(d)


def line_intersection(mu, d, mid, normal):
    a = np.column_stack([d, -normal])
    b = mid - mu
    sol, *_ = np.linalg.lstsq(a, b, rcond=None)
    return mu + sol[0] * d


def fit_r2(points, degree):
    pts = np.asarray(points, dtype=float)
    x = pts[:, 0] - np.mean(pts[:, 0])
    y = pts[:, 1]
    coeff = np.polyfit(x, y, degree)
    yhat = np.polyval(coeff, x)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")


def distances_at(cache, jd_tdb):
    es = norm(vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    ev = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb))
    vs = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb) - vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    return es, ev, vs


def compute_iers_guideline_method(cache):
    ca_jd = find_closest(cache)
    c1, c4 = contact_range(cache)
    use_jds = cache["jd_tdb"][(cache["jd_tdb"] >= c1) & (cache["jd_tdb"] <= c4)]
    basis = fixed_geocenter_basis(cache, ca_jd)

    points = {}
    mus = {}
    dirs = {}
    track_rows = []

    for site in [SITE_A, SITE_B]:
        pts = np.array([ray_screen_point_arcsec(cache, site, jd, basis) for jd in use_jds], dtype=float)
        mu, d = pca_direction(pts)
        points[site["key"]] = pts
        mus[site["key"]] = mu
        dirs[site["key"]] = d
        angle = math.degrees(math.atan2(d[1], d[0]))
        track_rows.append({
            "track": site["label"],
            "lat": site["lat_deg"],
            "lon": site["lon_deg_east"],
            "angle_deg": angle,
            "linear_R2": fit_r2(pts, 1),
            "quad_R2": fit_r2(pts, 2),
            "cubic_R2": fit_r2(pts, 3),
            "parabolic_R2": fit_r2(pts, 2),
        })

    tangent = unit(dirs[SITE_A["key"]] + dirs[SITE_B["key"]])
    if tangent[0] < 0:
        tangent = -tangent
    normal = np.array([-tangent[1], tangent[0]])
    common_angle = math.degrees(math.atan2(tangent[1], tangent[0]))

    mid = 0.5 * (mus[SITE_A["key"]] + mus[SITE_B["key"]])
    aprime = line_intersection(mus[SITE_A["key"]], dirs[SITE_A["key"]], mid, normal)
    bprime = line_intersection(mus[SITE_B["key"]], dirs[SITE_B["key"]], mid, normal)

    abp_vec = bprime - aprime
    abp_arcsec = float(np.sqrt(np.sum(abp_vec * abp_vec)))
    sep_arcsec = abs(float(np.dot(abp_vec, normal)))

    es, ev, vs = distances_at(cache, ca_jd)
    abp_km = math.tan(abp_arcsec / ARCSEC_PER_RAD) * es
    baseline_proj_km = abp_km * ev / vs
    raw_phi = sep_arcsec * (ev / vs) * (EARTH_RADIUS_KM / baseline_proj_km)
    iau_phi = raw_phi * (es / AU_KM)
    delta = iau_phi - IAU_SOLAR_PARALLAX_ARCSEC

    theta_mean = 0.5 * (track_rows[0]["angle_deg"] + track_rows[1]["angle_deg"])
    delta_percent = 100.0 * delta / IAU_SOLAR_PARALLAX_ARCSEC

    rho_R_earth_arcsec = sep_arcsec * EARTH_RADIUS_KM / baseline_proj_km

    parallax = {
        "baseline_proj_km": baseline_proj_km,
        "A_prime_B_prime_arcsec": abp_arcsec,
        "A_prime_B_prime_km": abp_km,
        "sep_arcsec": sep_arcsec,
        "rho_R_earth_arcsec": rho_R_earth_arcsec,
        "raw_phi_arcsec": raw_phi,
        "IAU_phi_arcsec": iau_phi,
        "IAU_delta_arcsec": delta,
        "IAU_delta_percent": delta_percent,
        "common_angle_deg": common_angle,
        "theta_mean_deg": theta_mean,
        "D_ES_AU": es / AU_KM,
        "D_EV_D_VS": ev / vs,
        "CA_utc": utc_at(cache, ca_jd),
        "CA_jd_tdb": ca_jd,
    }
    return parallax, track_rows




def html_value(value, unit):
    if isinstance(value, (int, float, np.floating)) and np.isfinite(value):
        if unit == "km":
            return f"{value:,.6f}"
        if unit == "d":
            return f"{value:.12f}"
        if unit == "%":
            return f"{value:.6f}"
        return f"{value:.6f}"
    return str(value)


def build_report_rows(parallax, track_rows):
    parallax_rows = [
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_arcsec"], "arcsec"),
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_km"], "km"),
        ("ρ<sub>OBS</sub>", parallax["sep_arcsec"], "arcsec"),
        ("ρ<sub>R⊕</sub>", parallax["rho_R_earth_arcsec"], "arcsec"),
        ("B<sub>EFF,OBS</sub>", parallax["baseline_proj_km"], "km"),
        ("π<sub>OBS</sub>", parallax["raw_phi_arcsec"], "arcsec"),
        ("π<sub>IAU</sub>", parallax["IAU_phi_arcsec"], "arcsec"),
        ("Δπ", parallax["IAU_delta_arcsec"], "arcsec"),
        ("Δπ", parallax["IAU_delta_percent"], "%"),
    ]

    track_rows_out = []
    for tr in track_rows:
        short = "NP" if tr["track"].lower().startswith("north") else "MIA"
        track_rows_out.append((f"θ<sub>{short}</sub>", tr["angle_deg"], "deg", tr["linear_R2"], tr["quad_R2"], tr["cubic_R2"]))

    track_rows_out.append(("θ̄", parallax["theta_mean_deg"], "deg", "", "", ""))
    track_rows_out.append(("θ<sub>COMMON</sub>", parallax["common_angle_deg"], "deg", "", "", ""))

    time_rows = [
        ("UTC<sub>CA</sub>", parallax["CA_utc"], ""),
        ("JD<sub>TDB,CA</sub>", parallax["CA_jd_tdb"], "d"),
        ("D<sub>ES</sub>/AU", parallax["D_ES_AU"], ""),
        ("D<sub>EV</sub>/D<sub>VS</sub>", parallax["D_EV_D_VS"], ""),
    ]

    return parallax_rows, track_rows_out, time_rows


def html_table(title, headers, rows):
    html = []
    html.append(f"<div class='iers-title'>{title}</div>")
    html.append("<table class='iers-table'>")
    html.append("<thead><tr>")
    for h in headers:
        html.append(f"<th>{h}</th>")
    html.append("</tr></thead><tbody>")
    for row in rows:
        html.append("<tr>")
        for i, cell in enumerate(row):
            html.append(f"<td>{cell}</td>" if i == 0 else f"<td class='num'>{cell}</td>")
        html.append("</tr>")
    html.append("</tbody></table>")
    return "".join(html)


def display_html_report(parallax, track_rows, csv_path):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)

    parallax_html_rows = [[q, html_value(v, u), u] for q, v, u in parallax_rows]

    track_html_rows = []
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        track_html_rows.append([
            q,
            html_value(theta, unit),
            unit,
            "" if lin == "" else f"{float(lin):.9f}",
            "" if quad == "" else f"{float(quad):.9f}",
            "" if cubic == "" else f"{float(cubic):.9f}",
        ])

    time_html_rows = [[q, html_value(v, u), u] for q, v, u in time_rows]

    css = """
    <style>
    .iers-wrap {
        background:#000000;
        color:#f2f2f2;
        font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, "Liberation Mono", monospace;
        padding: 12px 14px;
        width: 720px;
        border: 1px solid #666;
    }
    .iers-title {
        color:#ffffff;
        font-weight:700;
        letter-spacing:0.05em;
        text-align:center;
        border-top:1px solid #aaa;
        border-bottom:1px solid #aaa;
        padding:6px 0;
        margin:10px 0 8px 0;
    }
    .iers-table {
        width:100%;
        border-collapse:collapse;
        margin-bottom:10px;
        table-layout:fixed;
    }
    .iers-table th {
        color:#ffffff;
        font-weight:700;
        border-bottom:1px solid #aaa;
        padding:5px 6px;
        text-align:left;
    }
    .iers-table td {
        color:#f2f2f2;
        padding:4px 6px;
        border-bottom:1px solid #333;
        white-space:nowrap;
    }
    .iers-table td.num {
        text-align:right;
        font-weight:700;
    }
    .iers-table sub {
        font-size:65%;
        line-height:0;
        vertical-align:sub;
    }
    .iers-foot {
        margin-top:10px;
        color:#ddd;
        font-size:12px;
        word-break:break-all;
    }
    </style>
    """

    html = css
    html += "<div class='iers-wrap'>"
    html += html_table("IERS PARALLAX SOLUTION", ["Quantity", "Value", "Units"], parallax_html_rows)
    html += html_table("TRACK GEOMETRY", ["Quantity", "Value", "Units", "R² lin", "R² quad", "R² cubic"], track_html_rows)
    html += html_table("REFERENCE GEOMETRY", ["Quantity", "Value", "Units"], time_html_rows)
    html += f"<div class='iers-foot'>CSV: {csv_path}</div>"
    html += "</div>"

    try:
        from IPython.display import HTML, display
        display(HTML(html))
        return True
    except Exception:
        return False


def plain_label(label):
    return label.replace("<sub>", "₍").replace("</sub>", "₎")


def print_rule(width=66):
    print("─" * width)


def print_plain_section(title, rows, width=66):
    print()
    print("═" * width)
    print(title.center(width))
    print("═" * width)
    print(f"{'Quantity':<22} {'Value':>20} {'Units':<10}")
    print_rule(width)
    for q, v, u in rows:
        q2 = plain_label(q)
        val = html_value(v, u)
        print(f"{q2:<22} {val:>20} {u:<10}")
    print_rule(width)


def print_plain_track_section(track_rows_out, width=66):
    print()
    print("═" * width)
    print("TRACK GEOMETRY".center(width))
    print("═" * width)
    print(f"{'Quantity':<18} {'θ':>12} {'Units':<8} {'R² lin':>12} {'R² cubic':>12}")
    print_rule(width)
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        q2 = plain_label(q)
        theta_s = html_value(theta, unit)
        lin_s = "" if lin == "" else f"{float(lin):.9f}"
        cubic_s = "" if cubic == "" else f"{float(cubic):.9f}"
        print(f"{q2:<18} {theta_s:>12} {unit:<8} {lin_s:>12} {cubic_s:>12}")
    print_rule(width)


def write_csv(parallax, track_rows, out_csv):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
    rows = []
    for q, v, u in parallax_rows:
        rows.append({"Section": "PARALLAX", "Quantity": q, "Value": v, "Unit": u})
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        rows.append({"Section": "TRACK", "Quantity": q, "Value": theta, "Unit": unit})
        if lin != "":
            rows.append({"Section": "TRACK", "Quantity": q + " R2_LINEAR", "Value": lin, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": q + " R2_QUADRATIC", "Value": quad, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": q + " R2_CUBIC", "Value": cubic, "Unit": ""})
    for q, v, u in time_rows:
        rows.append({"Section": "REFERENCE", "Quantity": q, "Value": v, "Unit": u})
    pd.DataFrame(rows).to_csv(out_csv, index=False, float_format="%.12f")


def main():
    print(f"CODE OUTPUT: {VERSION}")

    df = build_master_geocenter()
    cache = build_cache(df)
    parallax, track_rows = compute_iers_guideline_method(cache)

    out_dir = "/content/IERS_TN36_V01_MASTER_OUTPUT"
    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, f"{VERSION}_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv")
    write_csv(parallax, track_rows, out_csv)

    rendered = display_html_report(parallax, track_rows, out_csv)

    if not rendered:
        parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
        print_plain_section("IERS PARALLAX SOLUTION", parallax_rows)
        print_plain_track_section(track_rows_out)
        print_plain_section("REFERENCE GEOMETRY", time_rows)

    print()
    print(f"CSV: {out_csv}")
    print(datetime.now(LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S %z"))
    print(f"END OF CODE OUTPUT: {VERSION}")


if __name__ == "__main__":
    main()
# IERS-0003


CODE OUTPUT: IERS-0003



CSV: /content/IERS_TN36_V01_MASTER_OUTPUT/IERS-0003_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv
2026-07-08 19:23:52 -0500
END OF CODE OUTPUT: IERS-0003


In [ ]:
# %load V02-0029_ESO_EIS_V07_IERS_DERIVATION_NORTH_POLE_MIAMI.py
# V02-0029
# Audit reference: IERS-guideline explicit station GCRS derivation for North Pole vs Miami, using JPL geocenter vectors and ERFA/Astropy IERS transforms.

import sys
import math
import subprocess
from datetime import datetime, timezone, timedelta

VERSION = "V02-0029"

AU_KM = 149597870.7
ARCSEC_PER_RAD = 206264.80624709636
SUN_RADIUS_KM = 695700.0
VENUS_RADIUS_KM = 6051.8
EARTH_RADIUS_KM = 6378.137
IAU_SOLAR_PARALLAX_ARCSEC = 8.794148
LOCAL_TZ = timezone(timedelta(hours=-5))

START = "2012-Jun-05 20:00"
STOP = "2012-Jun-06 07:30"
STEP = "1m"

SITE_A = {
    "key": "NORTH_POLE",
    "label": "North Pole",
    "lon_deg_east": 0.0,
    "lat_deg": 90.0,
    "height_m": 0.0,
}

SITE_B = {
    "key": "MIAMI_FL",
    "label": "Miami FL",
    "lon_deg_east": -80.1918,
    "lat_deg": 25.7617,
    "height_m": 0.0,
}

try:
    import numpy as np
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas"])
    import numpy as np
    import pandas as pd

try:
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "scipy"])
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq

try:
    from astroquery.jplhorizons import Horizons
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astroquery"])
    from astroquery.jplhorizons import Horizons

try:
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astropy"])
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers

# Use bundled/fetched IERS tables when available; allow degraded mode in Colab if network table access is unavailable.
iers.conf.auto_download = True
iers.conf.iers_degraded_accuracy = "warn"


def norm(v):
    return float(np.sqrt(np.dot(v, v)))


def unit(v):
    n = norm(v)
    if n == 0.0:
        raise RuntimeError("Zero vector.")
    return np.asarray(v, dtype=float) / n


def angular_sep_arcsec(a, b):
    c = float(np.clip(np.dot(unit(a), unit(b)), -1.0, 1.0))
    return math.acos(c) * ARCSEC_PER_RAD


def horizons_geocenter_vectors(target_id, prefix):
    obj = Horizons(
        id=target_id,
        location="500@399",
        epochs={"start": START, "stop": STOP, "step": STEP},
    )
    vec = obj.vectors().to_pandas()
    out = pd.DataFrame()
    out["jd_tdb"] = vec["datetime_jd"].astype(float)
    out["utc"] = vec["datetime_str"].astype(str)
    out[f"{prefix}_x_km"] = vec["x"].astype(float) * AU_KM
    out[f"{prefix}_y_km"] = vec["y"].astype(float) * AU_KM
    out[f"{prefix}_z_km"] = vec["z"].astype(float) * AU_KM
    return out


def build_master_geocenter():
    print("Downloading GEOCENTER_SUN")
    sun = horizons_geocenter_vectors("10", "GEOCENTER_SUN")
    print("Downloading GEOCENTER_VENUS")
    venus = horizons_geocenter_vectors("299", "GEOCENTER_VENUS")
    return sun.merge(venus, on=["jd_tdb", "utc"], how="inner")


def build_cache(df):
    cache = {"jd_tdb": df["jd_tdb"].to_numpy(dtype=float), "utc": df["utc"].astype(str).tolist()}
    for col in df.columns:
        if col.endswith("_km"):
            cache[col] = CubicSpline(cache["jd_tdb"], df[col].to_numpy(dtype=float), bc_type="natural")
    return cache


def vec_at(cache, prefix, jd_tdb):
    return np.array([
        float(cache[f"{prefix}_x_km"](jd_tdb)),
        float(cache[f"{prefix}_y_km"](jd_tdb)),
        float(cache[f"{prefix}_z_km"](jd_tdb)),
    ], dtype=float)


def utc_at(cache, jd_tdb):
    try:
        return Time(jd_tdb, format="jd", scale="tdb").utc.iso.replace(" ", " ")
    except Exception:
        jds = cache["jd_tdb"]
        idx = int(np.argmin(np.abs(jds - jd_tdb)))
        return cache["utc"][idx]


def earth_location(site):
    # Astropy EarthLocation uses the ERFA/SOFA geodetic-to-geocentric implementation.
    # Geodetic longitude is east-positive, matching Horizons convention used in this project.
    return EarthLocation.from_geodetic(
        lon=site["lon_deg_east"] * u.deg,
        lat=site["lat_deg"] * u.deg,
        height=site["height_m"] * u.m,
    )


def observer_gcrs_km(site, jd_tdb):
    # IERS path: geodetic site -> ITRS -> GCRS, implemented through Astropy/ERFA using IERS EOP tables.
    # The time is supplied as TDB and internally converted by Astropy as required for Earth orientation.
    t = Time(jd_tdb, format="jd", scale="tdb")
    loc = earth_location(site)
    pos, _vel = loc.get_gcrs_posvel(t)
    return np.array([pos.x.to_value(u.km), pos.y.to_value(u.km), pos.z.to_value(u.km)], dtype=float)


def topocentric_body_vector(cache, site, body_prefix, jd_tdb):
    return vec_at(cache, body_prefix, jd_tdb) - observer_gcrs_km(site, jd_tdb)


def site_sep_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    return angular_sep_arcsec(sun, venus)


def geocenter_sep_arcsec(cache, jd_tdb):
    return angular_sep_arcsec(vec_at(cache, "GEOCENTER_SUN", jd_tdb), vec_at(cache, "GEOCENTER_VENUS", jd_tdb))


def angular_radii_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    rs = math.atan2(SUN_RADIUS_KM, norm(sun)) * ARCSEC_PER_RAD
    rv = math.atan2(VENUS_RADIUS_KM, norm(venus)) * ARCSEC_PER_RAD
    return rs, rv


def contact_function(cache, site, event, jd_tdb):
    sep = site_sep_arcsec(cache, site, jd_tdb)
    rs, rv = angular_radii_arcsec(cache, site, jd_tdb)
    threshold = rs + rv if event in ["C1", "C4"] else rs - rv
    return sep - threshold


def find_roots(cache, site, event):
    jds = cache["jd_tdb"]
    vals = np.array([contact_function(cache, site, event, jd) for jd in jds], dtype=float)
    roots = []
    for i in range(len(jds) - 1):
        if not np.isfinite(vals[i]) or not np.isfinite(vals[i + 1]):
            continue
        if vals[i] == 0.0:
            roots.append(float(jds[i]))
        elif vals[i] * vals[i + 1] < 0.0:
            roots.append(float(brentq(
                lambda x: contact_function(cache, site, event, x),
                jds[i],
                jds[i + 1],
                xtol=1e-13,
                rtol=1e-13,
                maxiter=100,
            )))
    return roots


def contact_range(cache):
    roots = []
    for site in [SITE_A, SITE_B]:
        roots.extend(find_roots(cache, site, "C1"))
        roots.extend(find_roots(cache, site, "C4"))
    if not roots:
        raise RuntimeError("No C1/C4 contacts found.")
    return min(roots), max(roots)


def find_closest(cache):
    jds = cache["jd_tdb"]
    vals = [geocenter_sep_arcsec(cache, jd) for jd in jds]
    i = int(np.argmin(vals))
    a = jds[max(0, i - 3)]
    b = jds[min(len(jds) - 1, i + 3)]
    res = minimize_scalar(
        lambda jd: geocenter_sep_arcsec(cache, jd),
        bounds=(a, b),
        method="bounded",
        options={"xatol": 1e-13},
    )
    return float(res.x)


def fixed_geocenter_basis(cache, jd_tdb):
    sun = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    n = unit(sun)
    ref = np.array([0.0, 0.0, 1.0])
    if norm(np.cross(ref, n)) < 1e-12:
        ref = np.array([1.0, 0.0, 0.0])
    xhat = unit(np.cross(ref, n))
    yhat = unit(np.cross(n, xhat))
    return n, xhat, yhat


def ray_screen_point_arcsec(cache, site, jd_tdb, basis):
    n, xhat, yhat = basis
    obs = observer_gcrs_km(site, jd_tdb)
    sun_geo = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    venus_geo = vec_at(cache, "GEOCENTER_VENUS", jd_tdb)
    ray = venus_geo - obs
    screen_point = sun_geo
    denom = float(np.dot(ray, n))
    if abs(denom) < 1e-14:
        raise RuntimeError("Ray nearly parallel to solar screen.")
    tau = float(np.dot(screen_point - obs, n) / denom)
    hit = obs + tau * ray
    screen_vec = hit - sun_geo
    es = norm(sun_geo)
    x = math.atan2(float(np.dot(screen_vec, xhat)), es) * ARCSEC_PER_RAD
    y = math.atan2(float(np.dot(screen_vec, yhat)), es) * ARCSEC_PER_RAD
    return np.array([x, y], dtype=float)


def pca_direction(points):
    pts = np.asarray(points, dtype=float)
    mu = pts.mean(axis=0)
    centered = pts - mu
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    d = vt[0]
    if d[0] < 0:
        d = -d
    return mu, unit(d)


def line_intersection(mu, d, mid, normal):
    a = np.column_stack([d, -normal])
    b = mid - mu
    sol, *_ = np.linalg.lstsq(a, b, rcond=None)
    return mu + sol[0] * d


def fit_r2(points, degree):
    pts = np.asarray(points, dtype=float)
    x = pts[:, 0] - np.mean(pts[:, 0])
    y = pts[:, 1]
    coeff = np.polyfit(x, y, degree)
    yhat = np.polyval(coeff, x)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")


def distances_at(cache, jd_tdb):
    es = norm(vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    ev = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb))
    vs = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb) - vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    return es, ev, vs


def compute_iers_guideline_method(cache):
    ca_jd = find_closest(cache)
    c1, c4 = contact_range(cache)
    use_jds = cache["jd_tdb"][(cache["jd_tdb"] >= c1) & (cache["jd_tdb"] <= c4)]
    basis = fixed_geocenter_basis(cache, ca_jd)

    points = {}
    mus = {}
    dirs = {}
    track_rows = []

    for site in [SITE_A, SITE_B]:
        pts = np.array([ray_screen_point_arcsec(cache, site, jd, basis) for jd in use_jds], dtype=float)
        mu, d = pca_direction(pts)
        points[site["key"]] = pts
        mus[site["key"]] = mu
        dirs[site["key"]] = d
        angle = math.degrees(math.atan2(d[1], d[0]))
        track_rows.append({
            "track": site["label"],
            "lat": site["lat_deg"],
            "lon": site["lon_deg_east"],
            "angle_deg": angle,
            "linear_R2": fit_r2(pts, 1),
            "quad_R2": fit_r2(pts, 2),
            "cubic_R2": fit_r2(pts, 3),
            "parabolic_R2": fit_r2(pts, 2),
        })

    tangent = unit(dirs[SITE_A["key"]] + dirs[SITE_B["key"]])
    if tangent[0] < 0:
        tangent = -tangent
    normal = np.array([-tangent[1], tangent[0]])
    common_angle = math.degrees(math.atan2(tangent[1], tangent[0]))

    mid = 0.5 * (mus[SITE_A["key"]] + mus[SITE_B["key"]])
    aprime = line_intersection(mus[SITE_A["key"]], dirs[SITE_A["key"]], mid, normal)
    bprime = line_intersection(mus[SITE_B["key"]], dirs[SITE_B["key"]], mid, normal)

    abp_vec = bprime - aprime
    abp_arcsec = float(np.sqrt(np.sum(abp_vec * abp_vec)))
    sep_arcsec = abs(float(np.dot(abp_vec, normal)))

    es, ev, vs = distances_at(cache, ca_jd)
    abp_km = math.tan(abp_arcsec / ARCSEC_PER_RAD) * es
    baseline_proj_km = abp_km * ev / vs
    raw_phi = sep_arcsec * (ev / vs) * (EARTH_RADIUS_KM / baseline_proj_km)
    iau_phi = raw_phi * (es / AU_KM)
    delta = iau_phi - IAU_SOLAR_PARALLAX_ARCSEC

    parallax = {
        "baseline_proj_km": baseline_proj_km,
        "A_prime_B_prime_km": abp_km,
        "sep_arcsec": sep_arcsec,
        "raw_phi_arcsec": raw_phi,
        "IAU_phi_arcsec": iau_phi,
        "IAU_delta_arcsec": delta,
        "common_angle_deg": common_angle,
        "CA_utc": utc_at(cache, ca_jd),
    }
    return parallax, track_rows


def main():
    print(f"CODE OUTPUT: {VERSION}")
    print()
    print("PAIR")
    print(f"{SITE_A['label']} vs {SITE_B['label']}")
    print()

    df = build_master_geocenter()
    cache = build_cache(df)
    parallax, track_rows = compute_iers_guideline_method(cache)

    print()
    print("IERS-GUIDELINE PARALLAX SUMMARY")
    print("baseline_proj_km  A_prime_B_prime_km  sep_arcsec  raw_phi  IAU_phi  delta  common_angle  CA_utc")
    print(
        f"{parallax['baseline_proj_km']:16,.6f}  "
        f"{parallax['A_prime_B_prime_km']:19,.6f}  "
        f"{parallax['sep_arcsec']:10.6f}  "
        f"{parallax['raw_phi_arcsec']:7.6f}  "
        f"{parallax['IAU_phi_arcsec']:7.6f}  "
        f"{parallax['IAU_delta_arcsec']:+.6f}  "
        f"{parallax['common_angle_deg']:12.6f}  "
        f"{parallax['CA_utc']}"
    )
    print()

    print("TRACK SUMMARY")
    print("track        lat       lon      angle_deg  linear_R2   quad_R2     cubic_R2    parabolic_R2")
    for row in track_rows:
        print(
            f"{row['track']:<10s} "
            f"{row['lat']:8.4f} "
            f"{row['lon']:9.4f} "
            f"{row['angle_deg']:10.6f} "
            f"{row['linear_R2']:.9f} "
            f"{row['quad_R2']:.9f} "
            f"{row['cubic_R2']:.9f} "
            f"{row['parabolic_R2']:.9f}"
        )
    print()

    print("IERS IMPLEMENTATION NOTE")
    print("Observer position is computed by Astropy/ERFA EarthLocation.get_gcrs_posvel, which applies the ITRS-to-GCRS path using IERS Earth-orientation data when available.")
    print("Solar-system body vectors are JPL Horizons geocenter vectors; the topocentric vectors are body_GCRS - observer_GCRS.")
    print()
    print("COLOMBIA TIME", datetime.now(LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S %z"))
    print(f"# {VERSION}")


if __name__ == "__main__":
    main()
# V02-0029


Saving V02-0029_ESO_EIS_V07_IERS_DERIVATION_NORTH_POLE_MIAMI.py to V02-0029_ESO_EIS_V07_IERS_DERIVATION_NORTH_POLE_MIAMI.py
CODE OUTPUT: V02-0029

PAIR
North Pole vs Miami FL


IERS-GUIDELINE PARALLAX SUMMARY
baseline_proj_km  A_prime_B_prime_km  sep_arcsec  raw_phi  IAU_phi  delta  common_angle  CA_utc
    4,197.408601        10,555.777283   14.342821  8.666390  8.794144  -0.000004      8.528541  2012-06-06 01:23:11.323

TRACK SUMMARY
track        lat       lon      angle_deg  linear_R2   quad_R2     cubic_R2    parabolic_R2
North Pole  90.0000    0.0000   8.486000 0.999999997 1.000000000 1.000000000 1.000000000
Miami FL    25.7617  -80.1918   8.571083 0.999983046 0.999999884 0.999999994 0.999999884

IERS IMPLEMENTATION NOTE
Observer position is computed by Astropy/ERFA EarthLocation.get_gcrs_posvel, which applies the ITRS-to-GCRS path using IERS Earth-orientation data when available.
Solar-system body vectors are JPL Horizons geocenter vectors; the topocentric vectors are body_GCRS -

In [ ]:
# %load IERS_0005_TN36_DUAL_PAIR_REPORT.py
# IERS-0005
# Audit reference: formatted IERS North Pole Miami report derived from validated V02-0029 implementation.

import os
import os
import sys
import math
import subprocess
from datetime import datetime, timezone, timedelta

VERSION = "IERS-0005"

AU_KM = 149597870.7
ARCSEC_PER_RAD = 206264.80624709636
SUN_RADIUS_KM = 695700.0
VENUS_RADIUS_KM = 6051.8
EARTH_RADIUS_KM = 6378.137
IAU_SOLAR_PARALLAX_ARCSEC = 8.794148
LOCAL_TZ = timezone(timedelta(hours=-5))

START = "2012-Jun-05 20:00"
STOP = "2012-Jun-06 07:30"
STEP = "1m"

SITE_A = {
    "key": "NORTH_POLE",
    "label": "North Pole",
    "lon_deg_east": 0.0,
    "lat_deg": 90.0,
    "height_m": 0.0,
}

SITE_B = {
    "key": "MIAMI_FL",
    "label": "Miami FL",
    "lon_deg_east": -80.1918,
    "lat_deg": 25.7617,
    "height_m": 0.0,
}

PAIR_RUNS = [
    {
        "title": "North Pole \u2192 Miami",
        "start": "2012-Jun-05 20:00",
        "stop": "2012-Jun-06 07:30",
        "step": "1m",
        "site_a": {
            "key": "NORTH_POLE",
            "label": "North Pole",
            "lon_deg_east": 0.0,
            "lat_deg": 90.0,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "MIAMI_FL",
            "label": "Miami FL",
            "lon_deg_east": -80.1918,
            "lat_deg": 25.7617,
            "height_m": 0.0,
        },
    },
    {
        "title": "Tahiti Point Venus \u2192 Vard\u00f8",
        "start": "1769-Jun-03 18:00",
        "stop": "1769-Jun-04 03:30",
        "step": "1m",
        "site_a": {
            "key": "TAHITI_POINT_VENUS",
            "label": "Tahiti Point Venus",
            "lon_deg_east": -149.4970,
            "lat_deg": -17.4950,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "VARDO_NORWAY",
            "label": "Vard\u00f8 Norway",
            "lon_deg_east": 31.1107,
            "lat_deg": 70.3706,
            "height_m": 0.0,
        },
    },
]

try:
    import numpy as np
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas"])
    import numpy as np
    import pandas as pd

try:
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "scipy"])
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq

try:
    from astroquery.jplhorizons import Horizons
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astroquery"])
    from astroquery.jplhorizons import Horizons

try:
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astropy"])
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers

# Use bundled/fetched IERS tables when available; allow degraded mode in Colab if network table access is unavailable.
iers.conf.auto_download = True
iers.conf.iers_degraded_accuracy = "warn"


def norm(v):
    return float(np.sqrt(np.dot(v, v)))


def unit(v):
    n = norm(v)
    if n == 0.0:
        raise RuntimeError("Zero vector.")
    return np.asarray(v, dtype=float) / n


def angular_sep_arcsec(a, b):
    c = float(np.clip(np.dot(unit(a), unit(b)), -1.0, 1.0))
    return math.acos(c) * ARCSEC_PER_RAD


def horizons_geocenter_vectors(target_id, prefix):
    obj = Horizons(
        id=target_id,
        location="500@399",
        epochs={"start": START, "stop": STOP, "step": STEP},
    )
    vec = obj.vectors().to_pandas()
    out = pd.DataFrame()
    out["jd_tdb"] = vec["datetime_jd"].astype(float)
    out["utc"] = vec["datetime_str"].astype(str)
    out[f"{prefix}_x_km"] = vec["x"].astype(float) * AU_KM
    out[f"{prefix}_y_km"] = vec["y"].astype(float) * AU_KM
    out[f"{prefix}_z_km"] = vec["z"].astype(float) * AU_KM
    return out


def build_master_geocenter():
    sun = horizons_geocenter_vectors("10", "GEOCENTER_SUN")
    venus = horizons_geocenter_vectors("299", "GEOCENTER_VENUS")
    return sun.merge(venus, on=["jd_tdb", "utc"], how="inner")


def build_cache(df):
    cache = {"jd_tdb": df["jd_tdb"].to_numpy(dtype=float), "utc": df["utc"].astype(str).tolist()}
    for col in df.columns:
        if col.endswith("_km"):
            cache[col] = CubicSpline(cache["jd_tdb"], df[col].to_numpy(dtype=float), bc_type="natural")
    return cache


def vec_at(cache, prefix, jd_tdb):
    return np.array([
        float(cache[f"{prefix}_x_km"](jd_tdb)),
        float(cache[f"{prefix}_y_km"](jd_tdb)),
        float(cache[f"{prefix}_z_km"](jd_tdb)),
    ], dtype=float)


def utc_at(cache, jd_tdb):
    try:
        return Time(jd_tdb, format="jd", scale="tdb").utc.iso.replace(" ", " ")
    except Exception:
        jds = cache["jd_tdb"]
        idx = int(np.argmin(np.abs(jds - jd_tdb)))
        return cache["utc"][idx]


def earth_location(site):
    # Astropy EarthLocation uses the ERFA/SOFA geodetic-to-geocentric implementation.
    # Geodetic longitude is east-positive, matching Horizons convention used in this project.
    return EarthLocation.from_geodetic(
        lon=site["lon_deg_east"] * u.deg,
        lat=site["lat_deg"] * u.deg,
        height=site["height_m"] * u.m,
    )


def observer_gcrs_km(site, jd_tdb):
    # IERS path: geodetic site -> ITRS -> GCRS, implemented through Astropy/ERFA using IERS EOP tables.
    # The time is supplied as TDB and internally converted by Astropy as required for Earth orientation.
    t = Time(jd_tdb, format="jd", scale="tdb")
    loc = earth_location(site)
    pos, _vel = loc.get_gcrs_posvel(t)
    return np.array([pos.x.to_value(u.km), pos.y.to_value(u.km), pos.z.to_value(u.km)], dtype=float)


def topocentric_body_vector(cache, site, body_prefix, jd_tdb):
    return vec_at(cache, body_prefix, jd_tdb) - observer_gcrs_km(site, jd_tdb)


def site_sep_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    return angular_sep_arcsec(sun, venus)


def geocenter_sep_arcsec(cache, jd_tdb):
    return angular_sep_arcsec(vec_at(cache, "GEOCENTER_SUN", jd_tdb), vec_at(cache, "GEOCENTER_VENUS", jd_tdb))


def angular_radii_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    rs = math.atan2(SUN_RADIUS_KM, norm(sun)) * ARCSEC_PER_RAD
    rv = math.atan2(VENUS_RADIUS_KM, norm(venus)) * ARCSEC_PER_RAD
    return rs, rv


def contact_function(cache, site, event, jd_tdb):
    sep = site_sep_arcsec(cache, site, jd_tdb)
    rs, rv = angular_radii_arcsec(cache, site, jd_tdb)
    threshold = rs + rv if event in ["C1", "C4"] else rs - rv
    return sep - threshold


def find_roots(cache, site, event):
    jds = cache["jd_tdb"]
    vals = np.array([contact_function(cache, site, event, jd) for jd in jds], dtype=float)
    roots = []
    for i in range(len(jds) - 1):
        if not np.isfinite(vals[i]) or not np.isfinite(vals[i + 1]):
            continue
        if vals[i] == 0.0:
            roots.append(float(jds[i]))
        elif vals[i] * vals[i + 1] < 0.0:
            roots.append(float(brentq(
                lambda x: contact_function(cache, site, event, x),
                jds[i],
                jds[i + 1],
                xtol=1e-13,
                rtol=1e-13,
                maxiter=100,
            )))
    return roots


def contact_range(cache):
    roots = []
    for site in [SITE_A, SITE_B]:
        roots.extend(find_roots(cache, site, "C1"))
        roots.extend(find_roots(cache, site, "C4"))
    if not roots:
        raise RuntimeError("No C1/C4 contacts found.")
    return min(roots), max(roots)


def find_closest(cache):
    jds = cache["jd_tdb"]
    vals = [geocenter_sep_arcsec(cache, jd) for jd in jds]
    i = int(np.argmin(vals))
    a = jds[max(0, i - 3)]
    b = jds[min(len(jds) - 1, i + 3)]
    res = minimize_scalar(
        lambda jd: geocenter_sep_arcsec(cache, jd),
        bounds=(a, b),
        method="bounded",
        options={"xatol": 1e-13},
    )
    return float(res.x)


def fixed_geocenter_basis(cache, jd_tdb):
    sun = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    n = unit(sun)
    ref = np.array([0.0, 0.0, 1.0])
    if norm(np.cross(ref, n)) < 1e-12:
        ref = np.array([1.0, 0.0, 0.0])
    xhat = unit(np.cross(ref, n))
    yhat = unit(np.cross(n, xhat))
    return n, xhat, yhat


def ray_screen_point_arcsec(cache, site, jd_tdb, basis):
    n, xhat, yhat = basis
    obs = observer_gcrs_km(site, jd_tdb)
    sun_geo = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    venus_geo = vec_at(cache, "GEOCENTER_VENUS", jd_tdb)
    ray = venus_geo - obs
    screen_point = sun_geo
    denom = float(np.dot(ray, n))
    if abs(denom) < 1e-14:
        raise RuntimeError("Ray nearly parallel to solar screen.")
    tau = float(np.dot(screen_point - obs, n) / denom)
    hit = obs + tau * ray
    screen_vec = hit - sun_geo
    es = norm(sun_geo)
    x = math.atan2(float(np.dot(screen_vec, xhat)), es) * ARCSEC_PER_RAD
    y = math.atan2(float(np.dot(screen_vec, yhat)), es) * ARCSEC_PER_RAD
    return np.array([x, y], dtype=float)


def pca_direction(points):
    pts = np.asarray(points, dtype=float)
    mu = pts.mean(axis=0)
    centered = pts - mu
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    d = vt[0]
    if d[0] < 0:
        d = -d
    return mu, unit(d)


def line_intersection(mu, d, mid, normal):
    a = np.column_stack([d, -normal])
    b = mid - mu
    sol, *_ = np.linalg.lstsq(a, b, rcond=None)
    return mu + sol[0] * d


def fit_r2(points, degree):
    pts = np.asarray(points, dtype=float)
    x = pts[:, 0] - np.mean(pts[:, 0])
    y = pts[:, 1]
    coeff = np.polyfit(x, y, degree)
    yhat = np.polyval(coeff, x)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")


def distances_at(cache, jd_tdb):
    es = norm(vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    ev = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb))
    vs = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb) - vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    return es, ev, vs


def compute_iers_guideline_method(cache):
    ca_jd = find_closest(cache)
    c1, c4 = contact_range(cache)
    use_jds = cache["jd_tdb"][(cache["jd_tdb"] >= c1) & (cache["jd_tdb"] <= c4)]
    basis = fixed_geocenter_basis(cache, ca_jd)

    points = {}
    mus = {}
    dirs = {}
    track_rows = []

    for site in [SITE_A, SITE_B]:
        pts = np.array([ray_screen_point_arcsec(cache, site, jd, basis) for jd in use_jds], dtype=float)
        mu, d = pca_direction(pts)
        points[site["key"]] = pts
        mus[site["key"]] = mu
        dirs[site["key"]] = d
        angle = math.degrees(math.atan2(d[1], d[0]))
        track_rows.append({
            "track": site["label"],
            "lat": site["lat_deg"],
            "lon": site["lon_deg_east"],
            "angle_deg": angle,
            "linear_R2": fit_r2(pts, 1),
            "quad_R2": fit_r2(pts, 2),
            "cubic_R2": fit_r2(pts, 3),
            "parabolic_R2": fit_r2(pts, 2),
        })

    tangent = unit(dirs[SITE_A["key"]] + dirs[SITE_B["key"]])
    if tangent[0] < 0:
        tangent = -tangent
    normal = np.array([-tangent[1], tangent[0]])
    common_angle = math.degrees(math.atan2(tangent[1], tangent[0]))

    mid = 0.5 * (mus[SITE_A["key"]] + mus[SITE_B["key"]])
    aprime = line_intersection(mus[SITE_A["key"]], dirs[SITE_A["key"]], mid, normal)
    bprime = line_intersection(mus[SITE_B["key"]], dirs[SITE_B["key"]], mid, normal)

    abp_vec = bprime - aprime
    abp_arcsec = float(np.sqrt(np.sum(abp_vec * abp_vec)))
    sep_arcsec = abs(float(np.dot(abp_vec, normal)))

    es, ev, vs = distances_at(cache, ca_jd)
    abp_km = math.tan(abp_arcsec / ARCSEC_PER_RAD) * es
    baseline_proj_km = abp_km * ev / vs
    raw_phi = sep_arcsec * (ev / vs) * (EARTH_RADIUS_KM / baseline_proj_km)
    iau_phi = raw_phi * (es / AU_KM)
    delta = iau_phi - IAU_SOLAR_PARALLAX_ARCSEC

    theta_mean = 0.5 * (track_rows[0]["angle_deg"] + track_rows[1]["angle_deg"])
    delta_percent = 100.0 * delta / IAU_SOLAR_PARALLAX_ARCSEC

    rho_R_earth_arcsec = sep_arcsec * EARTH_RADIUS_KM / baseline_proj_km

    parallax = {
        "baseline_proj_km": baseline_proj_km,
        "A_prime_B_prime_arcsec": abp_arcsec,
        "A_prime_B_prime_km": abp_km,
        "sep_arcsec": sep_arcsec,
        "rho_R_earth_arcsec": rho_R_earth_arcsec,
        "raw_phi_arcsec": raw_phi,
        "IAU_phi_arcsec": iau_phi,
        "IAU_delta_arcsec": delta,
        "IAU_delta_percent": delta_percent,
        "common_angle_deg": common_angle,
        "theta_mean_deg": theta_mean,
        "D_ES_AU": es / AU_KM,
        "D_EV_D_VS": ev / vs,
        "CA_utc": utc_at(cache, ca_jd),
        "CA_jd_tdb": ca_jd,
    }
    return parallax, track_rows




def html_value(value, unit):
    if isinstance(value, (int, float, np.floating)) and np.isfinite(value):
        if unit == "km":
            return f"{value:,.6f}"
        if unit == "d":
            return f"{value:.12f}"
        if unit == "%":
            return f"{value:.6f}"
        return f"{value:.6f}"
    return str(value)


def build_report_rows(parallax, track_rows):
    parallax_rows = [
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_arcsec"], "arcsec"),
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_km"], "km"),
        ("ρ<sub>OBS</sub>", parallax["sep_arcsec"], "arcsec"),
        ("ρ<sub>R⊕</sub>", parallax["rho_R_earth_arcsec"], "arcsec"),
        ("B<sub>EFF,OBS</sub>", parallax["baseline_proj_km"], "km"),
        ("π<sub>OBS</sub>", parallax["raw_phi_arcsec"], "arcsec"),
        ("π<sub>IAU</sub>", parallax["IAU_phi_arcsec"], "arcsec"),
        ("Δπ", parallax["IAU_delta_arcsec"], "arcsec"),
        ("Δπ", parallax["IAU_delta_percent"], "%"),
    ]

    track_rows_out = []
    for tr in track_rows:
        short = "NP" if tr["track"].lower().startswith("north") else "MIA"
        track_rows_out.append((f"θ<sub>{short}</sub>", tr["angle_deg"], "deg", tr["linear_R2"], tr["quad_R2"], tr["cubic_R2"]))

    track_rows_out.append(("θ̄", parallax["theta_mean_deg"], "deg", "", "", ""))
    track_rows_out.append(("θ<sub>COMMON</sub>", parallax["common_angle_deg"], "deg", "", "", ""))

    time_rows = [
        ("UTC<sub>CA</sub>", parallax["CA_utc"], ""),
        ("JD<sub>TDB,CA</sub>", parallax["CA_jd_tdb"], "d"),
        ("D<sub>ES</sub>/AU", parallax["D_ES_AU"], ""),
        ("D<sub>EV</sub>/D<sub>VS</sub>", parallax["D_EV_D_VS"], ""),
    ]

    return parallax_rows, track_rows_out, time_rows


def html_table(title, headers, rows):
    html = []
    html.append(f"<div class='iers-title'>{title}</div>")
    html.append("<table class='iers-table'>")
    html.append("<thead><tr>")
    for h in headers:
        html.append(f"<th>{h}</th>")
    html.append("</tr></thead><tbody>")
    for row in rows:
        html.append("<tr>")
        for i, cell in enumerate(row):
            html.append(f"<td>{cell}</td>" if i == 0 else f"<td class='num'>{cell}</td>")
        html.append("</tr>")
    html.append("</tbody></table>")
    return "".join(html)


def display_html_report(parallax, track_rows, csv_path, pair_title):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)

    parallax_html_rows = [[q, html_value(v, u), u] for q, v, u in parallax_rows]

    track_html_rows = []
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        track_html_rows.append([
            q,
            html_value(theta, unit),
            unit,
            "" if lin == "" else f"{float(lin):.9f}",
            "" if quad == "" else f"{float(quad):.9f}",
            "" if cubic == "" else f"{float(cubic):.9f}",
        ])

    time_html_rows = [[q, html_value(v, u), u] for q, v, u in time_rows]

    css = """
    <style>
    .iers-wrap {
        background:#000000;
        color:#f2f2f2;
        font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, "Liberation Mono", monospace;
        padding: 12px 14px;
        width: 720px;
        border: 1px solid #666;
    }
    .iers-title {
        color:#ffffff;
        font-weight:700;
        letter-spacing:0.05em;
        text-align:center;
        border-top:1px solid #aaa;
        border-bottom:1px solid #aaa;
        padding:6px 0;
        margin:10px 0 8px 0;
    }
    .iers-table {
        width:100%;
        border-collapse:collapse;
        margin-bottom:10px;
        table-layout:fixed;
    }
    .iers-table th {
        color:#ffffff;
        font-weight:700;
        border-bottom:1px solid #aaa;
        padding:5px 6px;
        text-align:left;
    }
    .iers-table td {
        color:#f2f2f2;
        padding:4px 6px;
        border-bottom:1px solid #333;
        white-space:nowrap;
    }
    .iers-table td.num {
        text-align:right;
        font-weight:700;
    }
    .iers-table sub {
        font-size:65%;
        line-height:0;
        vertical-align:sub;
    }
    .iers-foot {
        margin-top:10px;
        color:#ddd;
        font-size:12px;
        word-break:break-all;
    }
    </style>
    """

    html = css
    html += "<div class='iers-wrap'>"
    html += html_table(f"IERS PARALLAX SOLUTION — {pair_title}", ["Quantity", "Value", "Units"], parallax_html_rows)
    html += html_table("TRACK GEOMETRY", ["Quantity", "Value", "Units", "R² lin", "R² quad", "R² cubic"], track_html_rows)
    html += html_table("REFERENCE GEOMETRY", ["Quantity", "Value", "Units"], time_html_rows)
    html += f"<div class='iers-foot'>CSV: {csv_path}</div>"
    html += "</div>"

    try:
        from IPython.display import HTML, display
        display(HTML(html))
        return True
    except Exception:
        return False


def plain_label(label):
    return label.replace("<sub>", "₍").replace("</sub>", "₎")


def print_rule(width=66):
    print("─" * width)


def print_plain_section(title, rows, width=66):
    print()
    print("═" * width)
    print(title.center(width))
    print("═" * width)
    print(f"{'Quantity':<22} {'Value':>20} {'Units':<10}")
    print_rule(width)
    for q, v, u in rows:
        q2 = plain_label(q)
        val = html_value(v, u)
        print(f"{q2:<22} {val:>20} {u:<10}")
    print_rule(width)


def print_plain_track_section(track_rows_out, width=66):
    print()
    print("═" * width)
    print("TRACK GEOMETRY".center(width))
    print("═" * width)
    print(f"{'Quantity':<18} {'θ':>12} {'Units':<8} {'R² lin':>12} {'R² cubic':>12}")
    print_rule(width)
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        q2 = plain_label(q)
        theta_s = html_value(theta, unit)
        lin_s = "" if lin == "" else f"{float(lin):.9f}"
        cubic_s = "" if cubic == "" else f"{float(cubic):.9f}"
        print(f"{q2:<18} {theta_s:>12} {unit:<8} {lin_s:>12} {cubic_s:>12}")
    print_rule(width)


def write_csv(parallax, track_rows, out_csv):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
    rows = []
    for q, v, u in parallax_rows:
        rows.append({"Section": "PARALLAX", "Quantity": q, "Value": v, "Unit": u})
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        rows.append({"Section": "TRACK", "Quantity": q, "Value": theta, "Unit": unit})
        if lin != "":
            rows.append({"Section": "TRACK", "Quantity": q + " R2_LINEAR", "Value": lin, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": q + " R2_QUADRATIC", "Value": quad, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": q + " R2_CUBIC", "Value": cubic, "Unit": ""})
    for q, v, u in time_rows:
        rows.append({"Section": "REFERENCE", "Quantity": q, "Value": v, "Unit": u})
    pd.DataFrame(rows).to_csv(out_csv, index=False, float_format="%.12f")



def run_pair(pair, out_dir):
    global START, STOP, STEP, SITE_A, SITE_B

    START = pair["start"]
    STOP = pair["stop"]
    STEP = pair["step"]
    SITE_A = pair["site_a"]
    SITE_B = pair["site_b"]

    df = build_master_geocenter()
    cache = build_cache(df)
    parallax, track_rows = compute_iers_guideline_method(cache)

    safe_title = (
        pair["title"]
        .replace(" ", "_")
        .replace("\u2192", "TO")
        .replace("\u00f8", "o")
        .replace("\u00d8", "O")
        .replace("/", "_")
    )

    out_csv = os.path.join(
        out_dir,
        f"{VERSION}_{safe_title}_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv",
    )
    write_csv(parallax, track_rows, out_csv)

    rendered = display_html_report(parallax, track_rows, out_csv, pair["title"])

    if not rendered:
        parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
        print_plain_section(f"IERS PARALLAX SOLUTION — {pair['title']}", parallax_rows)
        print_plain_track_section(track_rows_out)
        print_plain_section("REFERENCE GEOMETRY", time_rows)

    print()
    print(f"CSV: {out_csv}")


def main():
    print(f"CODE OUTPUT: {VERSION}")

    out_dir = "/content/IERS_TN36_V01_MASTER_OUTPUT"
    os.makedirs(out_dir, exist_ok=True)

    for pair in PAIR_RUNS:
        run_pair(pair, out_dir)

    print(datetime.now(LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S %z"))
    print(f"END OF CODE OUTPUT: {VERSION}")


if __name__ == "__main__":
    main()
# IERS-0005


Saving IERS_0005_TN36_DUAL_PAIR_REPORT.py to IERS_0005_TN36_DUAL_PAIR_REPORT.py
CODE OUTPUT: IERS-0005



CSV: /content/IERS_TN36_V01_MASTER_OUTPUT/IERS-0005_North_Pole_TO_Miami_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv



CSV: /content/IERS_TN36_V01_MASTER_OUTPUT/IERS-0005_Tahiti_Point_Venus_TO_Vardo_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv
2026-07-08 19:32:30 -0500
END OF CODE OUTPUT: IERS-0005


In [ ]:
# %load IERS_0006_TN36_THREE_PAIR_REPORT.py
# IERS-0006
# Audit reference: formatted IERS North Pole Miami report derived from validated V02-0029 implementation.

import os
import os
import os
import sys
import math
import subprocess
from datetime import datetime, timezone, timedelta

VERSION = "IERS-0006"

AU_KM = 149597870.7
ARCSEC_PER_RAD = 206264.80624709636
SUN_RADIUS_KM = 695700.0
VENUS_RADIUS_KM = 6051.8
EARTH_RADIUS_KM = 6378.137
IAU_SOLAR_PARALLAX_ARCSEC = 8.794148
LOCAL_TZ = timezone(timedelta(hours=-5))

START = "2012-Jun-05 20:00"
STOP = "2012-Jun-06 07:30"
STEP = "1m"

SITE_A = {
    "key": "NORTH_POLE",
    "label": "North Pole",
    "lon_deg_east": 0.0,
    "lat_deg": 90.0,
    "height_m": 0.0,
}

SITE_B = {
    "key": "MIAMI_FL",
    "label": "Miami FL",
    "lon_deg_east": -80.1918,
    "lat_deg": 25.7617,
    "height_m": 0.0,
}

PAIR_RUNS = [
    {
        "title": "North Pole \u2192 Miami",
        "start": "2012-Jun-05 20:00",
        "stop": "2012-Jun-06 07:30",
        "step": "1m",
        "site_a": {
            "key": "NORTH_POLE",
            "label": "North Pole",
            "lon_deg_east": 0.0,
            "lat_deg": 90.0,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "MIAMI_FL",
            "label": "Miami FL",
            "lon_deg_east": -80.1918,
            "lat_deg": 25.7617,
            "height_m": 0.0,
        },
    },
    {
        "title": "Tahiti Point Venus \u2192 Vard\u00f8",
        "start": "1769-Jun-03 18:00",
        "stop": "1769-Jun-04 03:30",
        "step": "1m",
        "site_a": {
            "key": "TAHITI_POINT_VENUS",
            "label": "Tahiti Point Venus",
            "lon_deg_east": -149.4970,
            "lat_deg": -17.4950,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "VARDO_NORWAY",
            "label": "Vard\u00f8 Norway",
            "lon_deg_east": 31.1107,
            "lat_deg": 70.3706,
            "height_m": 0.0,
        },
    },
    {
        "title": "Cape Town \u2192 Reykjavík",
        "start": "2012-Jun-05 20:00",
        "stop": "2012-Jun-06 07:30",
        "step": "1m",
        "site_a": {
            "key": "CAPE_TOWN",
            "label": "Cape Town",
            "lon_deg_east": 18.4241,
            "lat_deg": -33.9249,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "REYKJAVIK",
            "label": "Reykjavík",
            "lon_deg_east": -21.9426,
            "lat_deg": 64.1466,
            "height_m": 0.0,
        },
    },
]

try:
    import numpy as np
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas"])
    import numpy as np
    import pandas as pd

try:
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "scipy"])
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq

try:
    from astroquery.jplhorizons import Horizons
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astroquery"])
    from astroquery.jplhorizons import Horizons

try:
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astropy"])
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers

# Use bundled/fetched IERS tables when available; allow degraded mode in Colab if network table access is unavailable.
iers.conf.auto_download = True
iers.conf.iers_degraded_accuracy = "warn"


def norm(v):
    return float(np.sqrt(np.dot(v, v)))


def unit(v):
    n = norm(v)
    if n == 0.0:
        raise RuntimeError("Zero vector.")
    return np.asarray(v, dtype=float) / n


def angular_sep_arcsec(a, b):
    c = float(np.clip(np.dot(unit(a), unit(b)), -1.0, 1.0))
    return math.acos(c) * ARCSEC_PER_RAD


def horizons_geocenter_vectors(target_id, prefix):
    obj = Horizons(
        id=target_id,
        location="500@399",
        epochs={"start": START, "stop": STOP, "step": STEP},
    )
    vec = obj.vectors().to_pandas()
    out = pd.DataFrame()
    out["jd_tdb"] = vec["datetime_jd"].astype(float)
    out["utc"] = vec["datetime_str"].astype(str)
    out[f"{prefix}_x_km"] = vec["x"].astype(float) * AU_KM
    out[f"{prefix}_y_km"] = vec["y"].astype(float) * AU_KM
    out[f"{prefix}_z_km"] = vec["z"].astype(float) * AU_KM
    return out


def build_master_geocenter():
    sun = horizons_geocenter_vectors("10", "GEOCENTER_SUN")
    venus = horizons_geocenter_vectors("299", "GEOCENTER_VENUS")
    return sun.merge(venus, on=["jd_tdb", "utc"], how="inner")


def build_cache(df):
    cache = {"jd_tdb": df["jd_tdb"].to_numpy(dtype=float), "utc": df["utc"].astype(str).tolist()}
    for col in df.columns:
        if col.endswith("_km"):
            cache[col] = CubicSpline(cache["jd_tdb"], df[col].to_numpy(dtype=float), bc_type="natural")
    return cache


def vec_at(cache, prefix, jd_tdb):
    return np.array([
        float(cache[f"{prefix}_x_km"](jd_tdb)),
        float(cache[f"{prefix}_y_km"](jd_tdb)),
        float(cache[f"{prefix}_z_km"](jd_tdb)),
    ], dtype=float)


def utc_at(cache, jd_tdb):
    try:
        return Time(jd_tdb, format="jd", scale="tdb").utc.iso.replace(" ", " ")
    except Exception:
        jds = cache["jd_tdb"]
        idx = int(np.argmin(np.abs(jds - jd_tdb)))
        return cache["utc"][idx]


def earth_location(site):
    # Astropy EarthLocation uses the ERFA/SOFA geodetic-to-geocentric implementation.
    # Geodetic longitude is east-positive, matching Horizons convention used in this project.
    return EarthLocation.from_geodetic(
        lon=site["lon_deg_east"] * u.deg,
        lat=site["lat_deg"] * u.deg,
        height=site["height_m"] * u.m,
    )


def observer_gcrs_km(site, jd_tdb):
    # IERS path: geodetic site -> ITRS -> GCRS, implemented through Astropy/ERFA using IERS EOP tables.
    # The time is supplied as TDB and internally converted by Astropy as required for Earth orientation.
    t = Time(jd_tdb, format="jd", scale="tdb")
    loc = earth_location(site)
    pos, _vel = loc.get_gcrs_posvel(t)
    return np.array([pos.x.to_value(u.km), pos.y.to_value(u.km), pos.z.to_value(u.km)], dtype=float)


def topocentric_body_vector(cache, site, body_prefix, jd_tdb):
    return vec_at(cache, body_prefix, jd_tdb) - observer_gcrs_km(site, jd_tdb)


def site_sep_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    return angular_sep_arcsec(sun, venus)


def geocenter_sep_arcsec(cache, jd_tdb):
    return angular_sep_arcsec(vec_at(cache, "GEOCENTER_SUN", jd_tdb), vec_at(cache, "GEOCENTER_VENUS", jd_tdb))


def angular_radii_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    rs = math.atan2(SUN_RADIUS_KM, norm(sun)) * ARCSEC_PER_RAD
    rv = math.atan2(VENUS_RADIUS_KM, norm(venus)) * ARCSEC_PER_RAD
    return rs, rv


def contact_function(cache, site, event, jd_tdb):
    sep = site_sep_arcsec(cache, site, jd_tdb)
    rs, rv = angular_radii_arcsec(cache, site, jd_tdb)
    threshold = rs + rv if event in ["C1", "C4"] else rs - rv
    return sep - threshold


def find_roots(cache, site, event):
    jds = cache["jd_tdb"]
    vals = np.array([contact_function(cache, site, event, jd) for jd in jds], dtype=float)
    roots = []
    for i in range(len(jds) - 1):
        if not np.isfinite(vals[i]) or not np.isfinite(vals[i + 1]):
            continue
        if vals[i] == 0.0:
            roots.append(float(jds[i]))
        elif vals[i] * vals[i + 1] < 0.0:
            roots.append(float(brentq(
                lambda x: contact_function(cache, site, event, x),
                jds[i],
                jds[i + 1],
                xtol=1e-13,
                rtol=1e-13,
                maxiter=100,
            )))
    return roots


def contact_range(cache):
    roots = []
    for site in [SITE_A, SITE_B]:
        roots.extend(find_roots(cache, site, "C1"))
        roots.extend(find_roots(cache, site, "C4"))
    if not roots:
        raise RuntimeError("No C1/C4 contacts found.")
    return min(roots), max(roots)


def find_closest(cache):
    jds = cache["jd_tdb"]
    vals = [geocenter_sep_arcsec(cache, jd) for jd in jds]
    i = int(np.argmin(vals))
    a = jds[max(0, i - 3)]
    b = jds[min(len(jds) - 1, i + 3)]
    res = minimize_scalar(
        lambda jd: geocenter_sep_arcsec(cache, jd),
        bounds=(a, b),
        method="bounded",
        options={"xatol": 1e-13},
    )
    return float(res.x)


def fixed_geocenter_basis(cache, jd_tdb):
    sun = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    n = unit(sun)
    ref = np.array([0.0, 0.0, 1.0])
    if norm(np.cross(ref, n)) < 1e-12:
        ref = np.array([1.0, 0.0, 0.0])
    xhat = unit(np.cross(ref, n))
    yhat = unit(np.cross(n, xhat))
    return n, xhat, yhat


def ray_screen_point_arcsec(cache, site, jd_tdb, basis):
    n, xhat, yhat = basis
    obs = observer_gcrs_km(site, jd_tdb)
    sun_geo = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    venus_geo = vec_at(cache, "GEOCENTER_VENUS", jd_tdb)
    ray = venus_geo - obs
    screen_point = sun_geo
    denom = float(np.dot(ray, n))
    if abs(denom) < 1e-14:
        raise RuntimeError("Ray nearly parallel to solar screen.")
    tau = float(np.dot(screen_point - obs, n) / denom)
    hit = obs + tau * ray
    screen_vec = hit - sun_geo
    es = norm(sun_geo)
    x = math.atan2(float(np.dot(screen_vec, xhat)), es) * ARCSEC_PER_RAD
    y = math.atan2(float(np.dot(screen_vec, yhat)), es) * ARCSEC_PER_RAD
    return np.array([x, y], dtype=float)


def pca_direction(points):
    pts = np.asarray(points, dtype=float)
    mu = pts.mean(axis=0)
    centered = pts - mu
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    d = vt[0]
    if d[0] < 0:
        d = -d
    return mu, unit(d)


def line_intersection(mu, d, mid, normal):
    a = np.column_stack([d, -normal])
    b = mid - mu
    sol, *_ = np.linalg.lstsq(a, b, rcond=None)
    return mu + sol[0] * d


def fit_r2(points, degree):
    pts = np.asarray(points, dtype=float)
    x = pts[:, 0] - np.mean(pts[:, 0])
    y = pts[:, 1]
    coeff = np.polyfit(x, y, degree)
    yhat = np.polyval(coeff, x)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")


def distances_at(cache, jd_tdb):
    es = norm(vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    ev = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb))
    vs = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb) - vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    return es, ev, vs


def compute_iers_guideline_method(cache):
    ca_jd = find_closest(cache)
    c1, c4 = contact_range(cache)
    use_jds = cache["jd_tdb"][(cache["jd_tdb"] >= c1) & (cache["jd_tdb"] <= c4)]
    basis = fixed_geocenter_basis(cache, ca_jd)

    points = {}
    mus = {}
    dirs = {}
    track_rows = []

    for site in [SITE_A, SITE_B]:
        pts = np.array([ray_screen_point_arcsec(cache, site, jd, basis) for jd in use_jds], dtype=float)
        mu, d = pca_direction(pts)
        points[site["key"]] = pts
        mus[site["key"]] = mu
        dirs[site["key"]] = d
        angle = math.degrees(math.atan2(d[1], d[0]))
        track_rows.append({
            "track": site["label"],
            "lat": site["lat_deg"],
            "lon": site["lon_deg_east"],
            "angle_deg": angle,
            "linear_R2": fit_r2(pts, 1),
            "quad_R2": fit_r2(pts, 2),
            "cubic_R2": fit_r2(pts, 3),
            "parabolic_R2": fit_r2(pts, 2),
        })

    tangent = unit(dirs[SITE_A["key"]] + dirs[SITE_B["key"]])
    if tangent[0] < 0:
        tangent = -tangent
    normal = np.array([-tangent[1], tangent[0]])
    common_angle = math.degrees(math.atan2(tangent[1], tangent[0]))

    mid = 0.5 * (mus[SITE_A["key"]] + mus[SITE_B["key"]])
    aprime = line_intersection(mus[SITE_A["key"]], dirs[SITE_A["key"]], mid, normal)
    bprime = line_intersection(mus[SITE_B["key"]], dirs[SITE_B["key"]], mid, normal)

    abp_vec = bprime - aprime
    abp_arcsec = float(np.sqrt(np.sum(abp_vec * abp_vec)))
    sep_arcsec = abs(float(np.dot(abp_vec, normal)))

    es, ev, vs = distances_at(cache, ca_jd)
    abp_km = math.tan(abp_arcsec / ARCSEC_PER_RAD) * es
    baseline_proj_km = abp_km * ev / vs
    raw_phi = sep_arcsec * (ev / vs) * (EARTH_RADIUS_KM / baseline_proj_km)
    iau_phi = raw_phi * (es / AU_KM)
    delta = iau_phi - IAU_SOLAR_PARALLAX_ARCSEC

    theta_mean = 0.5 * (track_rows[0]["angle_deg"] + track_rows[1]["angle_deg"])
    delta_percent = 100.0 * delta / IAU_SOLAR_PARALLAX_ARCSEC

    rho_R_earth_arcsec = sep_arcsec * EARTH_RADIUS_KM / baseline_proj_km

    parallax = {
        "baseline_proj_km": baseline_proj_km,
        "A_prime_B_prime_arcsec": abp_arcsec,
        "A_prime_B_prime_km": abp_km,
        "sep_arcsec": sep_arcsec,
        "rho_R_earth_arcsec": rho_R_earth_arcsec,
        "raw_phi_arcsec": raw_phi,
        "IAU_phi_arcsec": iau_phi,
        "IAU_delta_arcsec": delta,
        "IAU_delta_percent": delta_percent,
        "common_angle_deg": common_angle,
        "theta_mean_deg": theta_mean,
        "D_ES_AU": es / AU_KM,
        "D_EV_D_VS": ev / vs,
        "CA_utc": utc_at(cache, ca_jd),
        "CA_jd_tdb": ca_jd,
    }
    return parallax, track_rows




def html_value(value, unit):
    if isinstance(value, (int, float, np.floating)) and np.isfinite(value):
        if unit == "km":
            return f"{value:,.6f}"
        if unit == "d":
            return f"{value:.12f}"
        if unit == "%":
            return f"{value:.6f}"
        return f"{value:.6f}"
    return str(value)


def build_report_rows(parallax, track_rows):
    parallax_rows = [
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_arcsec"], "arcsec"),
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_km"], "km"),
        ("ρ<sub>OBS</sub>", parallax["sep_arcsec"], "arcsec"),
        ("ρ<sub>R⊕</sub>", parallax["rho_R_earth_arcsec"], "arcsec"),
        ("B<sub>EFF,OBS</sub>", parallax["baseline_proj_km"], "km"),
        ("π<sub>OBS</sub>", parallax["raw_phi_arcsec"], "arcsec"),
        ("π<sub>IAU</sub>", parallax["IAU_phi_arcsec"], "arcsec"),
        ("Δπ", parallax["IAU_delta_arcsec"], "arcsec"),
        ("Δπ", parallax["IAU_delta_percent"], "%"),
    ]

    track_rows_out = []
    for tr in track_rows:
        short = "NP" if tr["track"].lower().startswith("north") else "MIA"
        track_rows_out.append((f"θ<sub>{short}</sub>", tr["angle_deg"], "deg", tr["linear_R2"], tr["quad_R2"], tr["cubic_R2"]))

    track_rows_out.append(("θ̄", parallax["theta_mean_deg"], "deg", "", "", ""))
    track_rows_out.append(("θ<sub>COMMON</sub>", parallax["common_angle_deg"], "deg", "", "", ""))

    time_rows = [
        ("UTC<sub>CA</sub>", parallax["CA_utc"], ""),
        ("JD<sub>TDB,CA</sub>", parallax["CA_jd_tdb"], "d"),
        ("D<sub>ES</sub>/AU", parallax["D_ES_AU"], ""),
        ("D<sub>EV</sub>/D<sub>VS</sub>", parallax["D_EV_D_VS"], ""),
    ]

    return parallax_rows, track_rows_out, time_rows


def html_table(title, headers, rows):
    html = []
    html.append(f"<div class='iers-title'>{title}</div>")
    html.append("<table class='iers-table'>")
    html.append("<thead><tr>")
    for h in headers:
        html.append(f"<th>{h}</th>")
    html.append("</tr></thead><tbody>")
    for row in rows:
        html.append("<tr>")
        for i, cell in enumerate(row):
            html.append(f"<td>{cell}</td>" if i == 0 else f"<td class='num'>{cell}</td>")
        html.append("</tr>")
    html.append("</tbody></table>")
    return "".join(html)


def display_html_report(parallax, track_rows, csv_path, pair_title):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)

    parallax_html_rows = [[q, html_value(v, u), u] for q, v, u in parallax_rows]

    track_html_rows = []
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        track_html_rows.append([
            q,
            html_value(theta, unit),
            unit,
            "" if lin == "" else f"{float(lin):.9f}",
            "" if quad == "" else f"{float(quad):.9f}",
            "" if cubic == "" else f"{float(cubic):.9f}",
        ])

    time_html_rows = [[q, html_value(v, u), u] for q, v, u in time_rows]

    css = """
    <style>
    .iers-wrap {
        background:#000000;
        color:#f2f2f2;
        font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, "Liberation Mono", monospace;
        padding: 12px 14px;
        width: 720px;
        border: 1px solid #666;
    }
    .iers-title {
        color:#ffffff;
        font-weight:700;
        letter-spacing:0.05em;
        text-align:center;
        border-top:1px solid #aaa;
        border-bottom:1px solid #aaa;
        padding:6px 0;
        margin:10px 0 8px 0;
    }
    .iers-table {
        width:100%;
        border-collapse:collapse;
        margin-bottom:10px;
        table-layout:fixed;
    }
    .iers-table th {
        color:#ffffff;
        font-weight:700;
        border-bottom:1px solid #aaa;
        padding:5px 6px;
        text-align:left;
    }
    .iers-table td {
        color:#f2f2f2;
        padding:4px 6px;
        border-bottom:1px solid #333;
        white-space:nowrap;
    }
    .iers-table td.num {
        text-align:right;
        font-weight:700;
    }
    .iers-table sub {
        font-size:65%;
        line-height:0;
        vertical-align:sub;
    }
    .iers-foot {
        margin-top:10px;
        color:#ddd;
        font-size:12px;
        word-break:break-all;
    }
    </style>
    """

    html = css
    html += "<div class='iers-wrap'>"
    html += html_table(f"IERS PARALLAX SOLUTION — {pair_title}", ["Quantity", "Value", "Units"], parallax_html_rows)
    html += html_table("TRACK GEOMETRY", ["Quantity", "Value", "Units", "R² lin", "R² quad", "R² cubic"], track_html_rows)
    html += html_table("REFERENCE GEOMETRY", ["Quantity", "Value", "Units"], time_html_rows)
    html += f"<div class='iers-foot'>CSV: {csv_path}</div>"
    html += "</div>"

    try:
        from IPython.display import HTML, display
        display(HTML(html))
        return True
    except Exception:
        return False


def plain_label(label):
    return label.replace("<sub>", "₍").replace("</sub>", "₎")


def print_rule(width=66):
    print("─" * width)


def print_plain_section(title, rows, width=66):
    print()
    print("═" * width)
    print(title.center(width))
    print("═" * width)
    print(f"{'Quantity':<22} {'Value':>20} {'Units':<10}")
    print_rule(width)
    for q, v, u in rows:
        q2 = plain_label(q)
        val = html_value(v, u)
        print(f"{q2:<22} {val:>20} {u:<10}")
    print_rule(width)


def print_plain_track_section(track_rows_out, width=66):
    print()
    print("═" * width)
    print("TRACK GEOMETRY".center(width))
    print("═" * width)
    print(f"{'Quantity':<18} {'θ':>12} {'Units':<8} {'R² lin':>12} {'R² cubic':>12}")
    print_rule(width)
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        q2 = plain_label(q)
        theta_s = html_value(theta, unit)
        lin_s = "" if lin == "" else f"{float(lin):.9f}"
        cubic_s = "" if cubic == "" else f"{float(cubic):.9f}"
        print(f"{q2:<18} {theta_s:>12} {unit:<8} {lin_s:>12} {cubic_s:>12}")
    print_rule(width)


def write_csv(parallax, track_rows, out_csv):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
    rows = []
    for q, v, u in parallax_rows:
        rows.append({"Section": "PARALLAX", "Quantity": q, "Value": v, "Unit": u})
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        rows.append({"Section": "TRACK", "Quantity": q, "Value": theta, "Unit": unit})
        if lin != "":
            rows.append({"Section": "TRACK", "Quantity": q + " R2_LINEAR", "Value": lin, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": q + " R2_QUADRATIC", "Value": quad, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": q + " R2_CUBIC", "Value": cubic, "Unit": ""})
    for q, v, u in time_rows:
        rows.append({"Section": "REFERENCE", "Quantity": q, "Value": v, "Unit": u})
    pd.DataFrame(rows).to_csv(out_csv, index=False, float_format="%.12f")



def run_pair(pair, out_dir):
    global START, STOP, STEP, SITE_A, SITE_B

    START = pair["start"]
    STOP = pair["stop"]
    STEP = pair["step"]
    SITE_A = pair["site_a"]
    SITE_B = pair["site_b"]

    df = build_master_geocenter()
    cache = build_cache(df)
    parallax, track_rows = compute_iers_guideline_method(cache)

    safe_title = (
        pair["title"]
        .replace(" ", "_")
        .replace("\u2192", "TO")
        .replace("\u00f8", "o")
        .replace("\u00d8", "O")
        .replace("/", "_")
    )

    out_csv = os.path.join(
        out_dir,
        f"{VERSION}_{safe_title}_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv",
    )
    write_csv(parallax, track_rows, out_csv)

    rendered = display_html_report(parallax, track_rows, out_csv, pair["title"])

    if not rendered:
        parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
        print_plain_section(f"IERS PARALLAX SOLUTION — {pair['title']}", parallax_rows)
        print_plain_track_section(track_rows_out)
        print_plain_section("REFERENCE GEOMETRY", time_rows)

    print()
    print(f"CSV: {out_csv}")


def main():
    print(f"CODE OUTPUT: {VERSION}")

    out_dir = "/content/IERS_TN36_V01_MASTER_OUTPUT"
    os.makedirs(out_dir, exist_ok=True)

    for pair in PAIR_RUNS:
        run_pair(pair, out_dir)

    print(datetime.now(LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S %z"))
    print(f"END OF CODE OUTPUT: {VERSION}")


if __name__ == "__main__":
    main()
# IERS-0006


Saving IERS_0006_TN36_THREE_PAIR_REPORT.py to IERS_0006_TN36_THREE_PAIR_REPORT.py
CODE OUTPUT: IERS-0006



CSV: /content/IERS_TN36_V01_MASTER_OUTPUT/IERS-0006_North_Pole_TO_Miami_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv



CSV: /content/IERS_TN36_V01_MASTER_OUTPUT/IERS-0006_Tahiti_Point_Venus_TO_Vardo_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv



CSV: /content/IERS_TN36_V01_MASTER_OUTPUT/IERS-0006_Cape_Town_TO_Reykjavík_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv
2026-07-08 19:39:23 -0500
END OF CODE OUTPUT: IERS-0006


In [ ]:
# %load IERS_0008_TN36_THREE_PAIR_BLACK_REPORT.py
# IERS-0008
# Audit reference: formatted IERS North Pole Miami report derived from validated V02-0029 implementation.

import os
import os
import os
import os
import sys
import math
import subprocess
from datetime import datetime, timezone, timedelta

VERSION = "IERS-0008"

AU_KM = 149597870.7
ARCSEC_PER_RAD = 206264.80624709636
SUN_RADIUS_KM = 695700.0
VENUS_RADIUS_KM = 6051.8
EARTH_RADIUS_KM = 6378.137
IAU_SOLAR_PARALLAX_ARCSEC = 8.794148
LOCAL_TZ = timezone(timedelta(hours=-5))

START = "2012-Jun-05 20:00"
STOP = "2012-Jun-06 07:30"
STEP = "1m"

SITE_A = {
    "key": "NORTH_POLE",
    "label": "North Pole",
    "lon_deg_east": 0.0,
    "lat_deg": 90.0,
    "height_m": 0.0,
}

SITE_B = {
    "key": "MIAMI_FL",
    "label": "Miami FL",
    "lon_deg_east": -80.1918,
    "lat_deg": 25.7617,
    "height_m": 0.0,
}

PAIR_RUNS = [
    {
        "title": "North Pole \u2192 Miami",
        "start": "2012-Jun-05 20:00",
        "stop": "2012-Jun-06 07:30",
        "step": "1m",
        "site_a": {
            "key": "NORTH_POLE",
            "label": "North Pole",
            "lon_deg_east": 0.0,
            "lat_deg": 90.0,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "MIAMI_FL",
            "label": "Miami FL",
            "lon_deg_east": -80.1918,
            "lat_deg": 25.7617,
            "height_m": 0.0,
        },
    },
    {
        "title": "Tahiti Point Venus \u2192 Vard\u00f8",
        "start": "1769-Jun-03 18:00",
        "stop": "1769-Jun-04 03:30",
        "step": "1m",
        "site_a": {
            "key": "TAHITI_POINT_VENUS",
            "label": "Tahiti Point Venus",
            "lon_deg_east": -149.4970,
            "lat_deg": -17.4950,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "VARDO_NORWAY",
            "label": "Vard\u00f8 Norway",
            "lon_deg_east": 31.1107,
            "lat_deg": 70.3706,
            "height_m": 0.0,
        },
    },
    {
        "title": "Cape Town \u2192 Reykjavík",
        "start": "2012-Jun-05 20:00",
        "stop": "2012-Jun-06 07:30",
        "step": "1m",
        "site_a": {
            "key": "CAPE_TOWN",
            "label": "Cape Town",
            "lon_deg_east": 18.4241,
            "lat_deg": -33.9249,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "REYKJAVIK",
            "label": "Reykjavík",
            "lon_deg_east": -21.9426,
            "lat_deg": 64.1466,
            "height_m": 0.0,
        },
    },
]

try:
    import numpy as np
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas"])
    import numpy as np
    import pandas as pd

try:
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "scipy"])
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq

try:
    from astroquery.jplhorizons import Horizons
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astroquery"])
    from astroquery.jplhorizons import Horizons

try:
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astropy"])
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers

# Use bundled/fetched IERS tables when available; allow degraded mode in Colab if network table access is unavailable.
iers.conf.auto_download = True
iers.conf.iers_degraded_accuracy = "warn"


def norm(v):
    return float(np.sqrt(np.dot(v, v)))


def unit(v):
    n = norm(v)
    if n == 0.0:
        raise RuntimeError("Zero vector.")
    return np.asarray(v, dtype=float) / n


def angular_sep_arcsec(a, b):
    c = float(np.clip(np.dot(unit(a), unit(b)), -1.0, 1.0))
    return math.acos(c) * ARCSEC_PER_RAD


def horizons_geocenter_vectors(target_id, prefix):
    obj = Horizons(
        id=target_id,
        location="500@399",
        epochs={"start": START, "stop": STOP, "step": STEP},
    )
    vec = obj.vectors().to_pandas()
    out = pd.DataFrame()
    out["jd_tdb"] = vec["datetime_jd"].astype(float)
    out["utc"] = vec["datetime_str"].astype(str)
    out[f"{prefix}_x_km"] = vec["x"].astype(float) * AU_KM
    out[f"{prefix}_y_km"] = vec["y"].astype(float) * AU_KM
    out[f"{prefix}_z_km"] = vec["z"].astype(float) * AU_KM
    return out


def build_master_geocenter():
    sun = horizons_geocenter_vectors("10", "GEOCENTER_SUN")
    venus = horizons_geocenter_vectors("299", "GEOCENTER_VENUS")
    return sun.merge(venus, on=["jd_tdb", "utc"], how="inner")


def build_cache(df):
    cache = {"jd_tdb": df["jd_tdb"].to_numpy(dtype=float), "utc": df["utc"].astype(str).tolist()}
    for col in df.columns:
        if col.endswith("_km"):
            cache[col] = CubicSpline(cache["jd_tdb"], df[col].to_numpy(dtype=float), bc_type="natural")
    return cache


def vec_at(cache, prefix, jd_tdb):
    return np.array([
        float(cache[f"{prefix}_x_km"](jd_tdb)),
        float(cache[f"{prefix}_y_km"](jd_tdb)),
        float(cache[f"{prefix}_z_km"](jd_tdb)),
    ], dtype=float)


def utc_at(cache, jd_tdb):
    try:
        return Time(jd_tdb, format="jd", scale="tdb").utc.iso.replace(" ", " ")
    except Exception:
        jds = cache["jd_tdb"]
        idx = int(np.argmin(np.abs(jds - jd_tdb)))
        return cache["utc"][idx]


def earth_location(site):
    # Astropy EarthLocation uses the ERFA/SOFA geodetic-to-geocentric implementation.
    # Geodetic longitude is east-positive, matching Horizons convention used in this project.
    return EarthLocation.from_geodetic(
        lon=site["lon_deg_east"] * u.deg,
        lat=site["lat_deg"] * u.deg,
        height=site["height_m"] * u.m,
    )


def observer_gcrs_km(site, jd_tdb):
    # IERS path: geodetic site -> ITRS -> GCRS, implemented through Astropy/ERFA using IERS EOP tables.
    # The time is supplied as TDB and internally converted by Astropy as required for Earth orientation.
    t = Time(jd_tdb, format="jd", scale="tdb")
    loc = earth_location(site)
    pos, _vel = loc.get_gcrs_posvel(t)
    return np.array([pos.x.to_value(u.km), pos.y.to_value(u.km), pos.z.to_value(u.km)], dtype=float)


def topocentric_body_vector(cache, site, body_prefix, jd_tdb):
    return vec_at(cache, body_prefix, jd_tdb) - observer_gcrs_km(site, jd_tdb)


def site_sep_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    return angular_sep_arcsec(sun, venus)


def geocenter_sep_arcsec(cache, jd_tdb):
    return angular_sep_arcsec(vec_at(cache, "GEOCENTER_SUN", jd_tdb), vec_at(cache, "GEOCENTER_VENUS", jd_tdb))


def angular_radii_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    rs = math.atan2(SUN_RADIUS_KM, norm(sun)) * ARCSEC_PER_RAD
    rv = math.atan2(VENUS_RADIUS_KM, norm(venus)) * ARCSEC_PER_RAD
    return rs, rv


def contact_function(cache, site, event, jd_tdb):
    sep = site_sep_arcsec(cache, site, jd_tdb)
    rs, rv = angular_radii_arcsec(cache, site, jd_tdb)
    threshold = rs + rv if event in ["C1", "C4"] else rs - rv
    return sep - threshold


def find_roots(cache, site, event):
    jds = cache["jd_tdb"]
    vals = np.array([contact_function(cache, site, event, jd) for jd in jds], dtype=float)
    roots = []
    for i in range(len(jds) - 1):
        if not np.isfinite(vals[i]) or not np.isfinite(vals[i + 1]):
            continue
        if vals[i] == 0.0:
            roots.append(float(jds[i]))
        elif vals[i] * vals[i + 1] < 0.0:
            roots.append(float(brentq(
                lambda x: contact_function(cache, site, event, x),
                jds[i],
                jds[i + 1],
                xtol=1e-13,
                rtol=1e-13,
                maxiter=100,
            )))
    return roots


def contact_range(cache):
    roots = []
    for site in [SITE_A, SITE_B]:
        roots.extend(find_roots(cache, site, "C1"))
        roots.extend(find_roots(cache, site, "C4"))
    if not roots:
        raise RuntimeError("No C1/C4 contacts found.")
    return min(roots), max(roots)


def find_closest(cache):
    jds = cache["jd_tdb"]
    vals = [geocenter_sep_arcsec(cache, jd) for jd in jds]
    i = int(np.argmin(vals))
    a = jds[max(0, i - 3)]
    b = jds[min(len(jds) - 1, i + 3)]
    res = minimize_scalar(
        lambda jd: geocenter_sep_arcsec(cache, jd),
        bounds=(a, b),
        method="bounded",
        options={"xatol": 1e-13},
    )
    return float(res.x)


def fixed_geocenter_basis(cache, jd_tdb):
    sun = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    n = unit(sun)
    ref = np.array([0.0, 0.0, 1.0])
    if norm(np.cross(ref, n)) < 1e-12:
        ref = np.array([1.0, 0.0, 0.0])
    xhat = unit(np.cross(ref, n))
    yhat = unit(np.cross(n, xhat))
    return n, xhat, yhat


def ray_screen_point_arcsec(cache, site, jd_tdb, basis):
    n, xhat, yhat = basis
    obs = observer_gcrs_km(site, jd_tdb)
    sun_geo = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    venus_geo = vec_at(cache, "GEOCENTER_VENUS", jd_tdb)
    ray = venus_geo - obs
    screen_point = sun_geo
    denom = float(np.dot(ray, n))
    if abs(denom) < 1e-14:
        raise RuntimeError("Ray nearly parallel to solar screen.")
    tau = float(np.dot(screen_point - obs, n) / denom)
    hit = obs + tau * ray
    screen_vec = hit - sun_geo
    es = norm(sun_geo)
    x = math.atan2(float(np.dot(screen_vec, xhat)), es) * ARCSEC_PER_RAD
    y = math.atan2(float(np.dot(screen_vec, yhat)), es) * ARCSEC_PER_RAD
    return np.array([x, y], dtype=float)


def pca_direction(points):
    pts = np.asarray(points, dtype=float)
    mu = pts.mean(axis=0)
    centered = pts - mu
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    d = vt[0]
    if d[0] < 0:
        d = -d
    return mu, unit(d)


def line_intersection(mu, d, mid, normal):
    a = np.column_stack([d, -normal])
    b = mid - mu
    sol, *_ = np.linalg.lstsq(a, b, rcond=None)
    return mu + sol[0] * d


def fit_r2(points, degree):
    pts = np.asarray(points, dtype=float)
    x = pts[:, 0] - np.mean(pts[:, 0])
    y = pts[:, 1]
    coeff = np.polyfit(x, y, degree)
    yhat = np.polyval(coeff, x)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")


def distances_at(cache, jd_tdb):
    es = norm(vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    ev = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb))
    vs = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb) - vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    return es, ev, vs


def compute_iers_guideline_method(cache):
    ca_jd = find_closest(cache)
    c1, c4 = contact_range(cache)
    use_jds = cache["jd_tdb"][(cache["jd_tdb"] >= c1) & (cache["jd_tdb"] <= c4)]
    basis = fixed_geocenter_basis(cache, ca_jd)

    points = {}
    mus = {}
    dirs = {}
    track_rows = []

    for site in [SITE_A, SITE_B]:
        pts = np.array([ray_screen_point_arcsec(cache, site, jd, basis) for jd in use_jds], dtype=float)
        mu, d = pca_direction(pts)
        points[site["key"]] = pts
        mus[site["key"]] = mu
        dirs[site["key"]] = d
        angle = math.degrees(math.atan2(d[1], d[0]))
        track_rows.append({
            "track": site["label"],
            "lat": site["lat_deg"],
            "lon": site["lon_deg_east"],
            "angle_deg": angle,
            "linear_R2": fit_r2(pts, 1),
            "quad_R2": fit_r2(pts, 2),
            "cubic_R2": fit_r2(pts, 3),
            "parabolic_R2": fit_r2(pts, 2),
        })

    tangent = unit(dirs[SITE_A["key"]] + dirs[SITE_B["key"]])
    if tangent[0] < 0:
        tangent = -tangent
    normal = np.array([-tangent[1], tangent[0]])
    common_angle = math.degrees(math.atan2(tangent[1], tangent[0]))

    mid = 0.5 * (mus[SITE_A["key"]] + mus[SITE_B["key"]])
    aprime = line_intersection(mus[SITE_A["key"]], dirs[SITE_A["key"]], mid, normal)
    bprime = line_intersection(mus[SITE_B["key"]], dirs[SITE_B["key"]], mid, normal)

    abp_vec = bprime - aprime
    abp_arcsec = float(np.sqrt(np.sum(abp_vec * abp_vec)))
    sep_arcsec = abs(float(np.dot(abp_vec, normal)))

    es, ev, vs = distances_at(cache, ca_jd)
    abp_km = math.tan(abp_arcsec / ARCSEC_PER_RAD) * es
    baseline_proj_km = abp_km * ev / vs
    raw_phi = sep_arcsec * (ev / vs) * (EARTH_RADIUS_KM / baseline_proj_km)
    iau_phi = raw_phi * (es / AU_KM)
    delta = iau_phi - IAU_SOLAR_PARALLAX_ARCSEC

    theta_mean = 0.5 * (track_rows[0]["angle_deg"] + track_rows[1]["angle_deg"])
    delta_percent = 100.0 * delta / IAU_SOLAR_PARALLAX_ARCSEC

    rho_R_earth_arcsec = sep_arcsec * EARTH_RADIUS_KM / baseline_proj_km

    parallax = {
        "baseline_proj_km": baseline_proj_km,
        "A_prime_B_prime_arcsec": abp_arcsec,
        "A_prime_B_prime_km": abp_km,
        "sep_arcsec": sep_arcsec,
        "rho_R_earth_arcsec": rho_R_earth_arcsec,
        "raw_phi_arcsec": raw_phi,
        "IAU_phi_arcsec": iau_phi,
        "IAU_delta_arcsec": delta,
        "IAU_delta_percent": delta_percent,
        "common_angle_deg": common_angle,
        "theta_mean_deg": theta_mean,
        "D_ES_AU": es / AU_KM,
        "D_EV_D_VS": ev / vs,
        "CA_utc": utc_at(cache, ca_jd),
        "CA_jd_tdb": ca_jd,
    }
    return parallax, track_rows






def html_value(value, unit):
    if isinstance(value, (int, float, np.floating)) and np.isfinite(value):
        if unit == "km":
            return f"{value:,.6f}"
        if unit == "d":
            return f"{value:.12f}"
        if unit == "%":
            return f"{value:.6f}"
        return f"{value:.6f}"
    return str(value)


def site_short_label(label):
    name = str(label).strip()
    low = name.lower()
    if low.startswith("north"):
        return "NP"
    if low.startswith("miami"):
        return "MIA"
    if "tahiti" in low:
        return "TAHITI"
    if "vard" in low or "vård" in low or "vør" in low or "vø" in low:
        return "VARDO"
    if low.startswith("cape"):
        return "CAPE"
    if low.startswith("reyk"):
        return "REYK"
    return name.split()[0].upper()[:6]


def build_report_rows(parallax, track_rows):
    parallax_rows = [
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_arcsec"], "arcsec"),
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_km"], "km"),
        ("ρ<sub>OBS</sub>", parallax["sep_arcsec"], "arcsec"),
        ("ρ<sub>R⊕</sub>", parallax["rho_R_earth_arcsec"], "arcsec"),
        ("B<sub>eff,OBS</sub>", parallax["baseline_proj_km"], "km"),
        ("π<sub>OBS</sub>", parallax["raw_phi_arcsec"], "arcsec"),
        ("π<sub>IAU</sub>", parallax["IAU_phi_arcsec"], "arcsec"),
        ("Δπ", parallax["IAU_delta_arcsec"], "arcsec"),
        ("Δπ<sub>%</sub>", parallax["IAU_delta_percent"], "%"),
    ]

    track_rows_out = []
    for tr in track_rows:
        short = site_short_label(tr["track"])
        track_rows_out.append((f"θ<sub>{short}</sub>", tr["angle_deg"], "deg", tr["linear_R2"], tr["quad_R2"], tr["cubic_R2"]))

    track_rows_out.append(("θ̄", parallax["theta_mean_deg"], "deg", "", "", ""))
    track_rows_out.append(("θ<sub>COMMON</sub>", parallax["common_angle_deg"], "deg", "", "", ""))

    time_rows = [
        ("UTC<sub>CA</sub>", parallax["CA_utc"], ""),
        ("JD<sub>TDB,CA</sub>", parallax["CA_jd_tdb"], "d"),
        ("D<sub>ES</sub>/AU", parallax["D_ES_AU"], ""),
        ("D<sub>EV</sub>/D<sub>VS</sub>", parallax["D_EV_D_VS"], ""),
    ]

    return parallax_rows, track_rows_out, time_rows


def html_table(title, headers, rows, col_widths):
    html = []
    html.append(f"<div class='iers-title'>{title}</div>")
    html.append("<table class='iers-table'>")
    html.append("<colgroup>")
    for w in col_widths:
        html.append(f"<col style='width:{w};'>")
    html.append("</colgroup>")
    html.append("<thead><tr>")
    for i, h in enumerate(headers):
        cls = "num" if i == 1 or i >= 3 else ""
        html.append(f"<th class='{cls}'>{h}</th>")
    html.append("</tr></thead><tbody>")
    for row in rows:
        html.append("<tr>")
        for i, cell in enumerate(row):
            cls = "num" if i == 1 or i >= 3 else ""
            html.append(f"<td class='{cls}'>{cell}</td>")
        html.append("</tr>")
    html.append("</tbody></table>")
    return "".join(html)


def display_html_report(parallax, track_rows, csv_path, pair_title):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)

    parallax_html_rows = [[q, html_value(v, u), u] for q, v, u in parallax_rows]

    track_html_rows = []
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        track_html_rows.append([
            q,
            html_value(theta, unit),
            unit,
            "" if lin == "" else f"{float(lin):.9f}",
            "" if quad == "" else f"{float(quad):.9f}",
            "" if cubic == "" else f"{float(cubic):.9f}",
        ])

    time_html_rows = [[q, html_value(v, u), u] for q, v, u in time_rows]

    css = """
    <style>
    .iers-wrap {
        background:#000000;
        color:#f2f2f2;
        font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, "Liberation Mono", monospace;
        padding: 12px 14px;
        width: 760px;
        border: 1px solid #666;
        margin: 12px 0 18px 0;
    }
    .iers-title {
        color:#ffffff;
        font-weight:700;
        letter-spacing:0.05em;
        text-align:center;
        border-top:1px solid #aaa;
        border-bottom:1px solid #aaa;
        padding:6px 0;
        margin:10px 0 8px 0;
    }
    .iers-table {
        width:100%;
        border-collapse:collapse;
        margin-bottom:10px;
        table-layout:fixed;
    }
    .iers-table th {
        color:#ffffff;
        font-weight:700;
        border-bottom:1px solid #aaa;
        padding:5px 6px;
        text-align:left;
        white-space:nowrap;
    }
    .iers-table th.num {
        text-align:right;
    }
    .iers-table td {
        color:#f2f2f2;
        padding:4px 6px;
        border-bottom:1px solid #333;
        white-space:nowrap;
        text-align:left;
    }
    .iers-table td.num {
        text-align:right;
        font-weight:700;
        font-variant-numeric: tabular-nums;
    }
    .iers-table sub {
        font-size:65%;
        line-height:0;
        vertical-align:sub;
    }
    .iers-foot {
        margin-top:10px;
        color:#ddd;
        font-size:12px;
        word-break:break-all;
    }
    </style>
    """

    html = css
    html += "<div class='iers-wrap'>"
    html += html_table(
        f"IERS PARALLAX SOLUTION — {pair_title}",
        ["Quantity", "Value", "Units"],
        parallax_html_rows,
        ["34%", "42%", "24%"],
    )
    html += html_table(
        "TRACK GEOMETRY",
        ["Quantity", "Value", "Units", "R² lin", "R² quad", "R² cubic"],
        track_html_rows,
        ["18%", "18%", "12%", "17%", "17%", "18%"],
    )
    html += html_table(
        "REFERENCE GEOMETRY",
        ["Quantity", "Value", "Units"],
        time_html_rows,
        ["34%", "52%", "14%"],
    )
    html += f"<div class='iers-foot'>CSV: {csv_path}</div>"
    html += "</div>"

    try:
        from IPython.display import HTML, display
        display(HTML(html))
        return True
    except Exception:
        return False


def plain_label(label):
    return (
        str(label)
        .replace("<sub>", "₍")
        .replace("</sub>", "₎")
    )


def print_rule(width=74):
    print("─" * width)


def print_plain_section(title, rows, width=74):
    print()
    print("═" * width)
    print(title.center(width))
    print("═" * width)
    print(f"{'Quantity':<22} {'Value':>22} {'Units':<10}")
    print_rule(width)
    for q, v, u in rows:
        print(f"{plain_label(q):<22} {html_value(v, u):>22} {u:<10}")
    print_rule(width)


def print_plain_track_section(track_rows_out, width=74):
    print()
    print("═" * width)
    print("TRACK GEOMETRY".center(width))
    print("═" * width)
    print(f"{'Quantity':<18} {'Value':>12} {'Units':<8} {'R² lin':>12} {'R² cubic':>12}")
    print_rule(width)
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        lin_s = "" if lin == "" else f"{float(lin):.9f}"
        cubic_s = "" if cubic == "" else f"{float(cubic):.9f}"
        print(f"{plain_label(q):<18} {html_value(theta, unit):>12} {unit:<8} {lin_s:>12} {cubic_s:>12}")
    print_rule(width)


def write_csv(parallax, track_rows, out_csv):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
    rows = []
    for q, v, u in parallax_rows:
        rows.append({"Section": "PARALLAX", "Quantity": plain_label(q), "Value": v, "Unit": u})
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        rows.append({"Section": "TRACK", "Quantity": plain_label(q), "Value": theta, "Unit": unit})
        if lin != "":
            rows.append({"Section": "TRACK", "Quantity": plain_label(q) + " R2_LINEAR", "Value": lin, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": plain_label(q) + " R2_QUADRATIC", "Value": quad, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": plain_label(q) + " R2_CUBIC", "Value": cubic, "Unit": ""})
    for q, v, u in time_rows:
        rows.append({"Section": "REFERENCE", "Quantity": plain_label(q), "Value": v, "Unit": u})
    pd.DataFrame(rows).to_csv(out_csv, index=False, float_format="%.12f")


def run_pair(pair, out_dir):
    global START, STOP, STEP, SITE_A, SITE_B

    START = pair["start"]
    STOP = pair["stop"]
    STEP = pair["step"]
    SITE_A = pair["site_a"]
    SITE_B = pair["site_b"]

    df = build_master_geocenter()
    cache = build_cache(df)
    parallax, track_rows = compute_iers_guideline_method(cache)

    safe_title = (
        pair["title"]
        .replace(" ", "_")
        .replace("→", "TO")
        .replace("ø", "o")
        .replace("Ø", "O")
        .replace("í", "i")
        .replace("Í", "I")
        .replace("/", "_")
    )

    out_csv = os.path.join(
        out_dir,
        f"{VERSION}_{safe_title}_BLACK_SUBSCRIPT_ENGINEERING_REPORT.csv",
    )
    write_csv(parallax, track_rows, out_csv)

    rendered = display_html_report(parallax, track_rows, out_csv, pair["title"])

    if not rendered:
        parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
        print_plain_section(f"IERS PARALLAX SOLUTION — {pair['title']}", parallax_rows)
        print_plain_track_section(track_rows_out)
        print_plain_section("REFERENCE GEOMETRY", time_rows)
        print(f"CSV: {out_csv}")

def main():
    print(f"CODE OUTPUT: {VERSION}")

    out_dir = "/content/IERS_TN36_V01_MASTER_OUTPUT"
    os.makedirs(out_dir, exist_ok=True)

    for pair in PAIR_RUNS:
        run_pair(pair, out_dir)

    print(datetime.now(LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S %z"))
    print(f"END OF CODE OUTPUT: {VERSION}")


if __name__ == "__main__":
    main()
# IERS-0008


Saving IERS_0008_TN36_THREE_PAIR_BLACK_REPORT.py to IERS_0008_TN36_THREE_PAIR_BLACK_REPORT.py
CODE OUTPUT: IERS-0008


2026-07-08 19:50:22 -0500
END OF CODE OUTPUT: IERS-0008


In [ ]:
# %load IERS_0009_TN36_IAU_CONSTANT_AUDIT.py
# IERS-0009
# Audit reference: formatted IERS North Pole Miami report derived from validated V02-0029 implementation.

import os
import os
import os
import os
import sys
import math
import subprocess
from datetime import datetime, timezone, timedelta

VERSION = "IERS-0009"

AU_KM = 149597870.7
ARCSEC_PER_RAD = 206264.80624709636
SUN_RADIUS_KM = 695700.0
VENUS_RADIUS_KM = 6051.8
EARTH_RADIUS_KM = 6378.137
IAU_EARTH_EQUATORIAL_RADIUS_KM = 6378.1366
IAU_SOLAR_PARALLAX_ARCSEC = math.asin(IAU_EARTH_EQUATORIAL_RADIUS_KM / AU_KM) * ARCSEC_PER_RAD
LOCAL_TZ = timezone(timedelta(hours=-5))

START = "2012-Jun-05 20:00"
STOP = "2012-Jun-06 07:30"
STEP = "1m"

SITE_A = {
    "key": "NORTH_POLE",
    "label": "North Pole",
    "lon_deg_east": 0.0,
    "lat_deg": 90.0,
    "height_m": 0.0,
}

SITE_B = {
    "key": "MIAMI_FL",
    "label": "Miami FL",
    "lon_deg_east": -80.1918,
    "lat_deg": 25.7617,
    "height_m": 0.0,
}

PAIR_RUNS = [
    {
        "title": "North Pole \u2192 Miami",
        "start": "2012-Jun-05 20:00",
        "stop": "2012-Jun-06 07:30",
        "step": "1m",
        "site_a": {
            "key": "NORTH_POLE",
            "label": "North Pole",
            "lon_deg_east": 0.0,
            "lat_deg": 90.0,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "MIAMI_FL",
            "label": "Miami FL",
            "lon_deg_east": -80.1918,
            "lat_deg": 25.7617,
            "height_m": 0.0,
        },
    },
    {
        "title": "Tahiti Point Venus \u2192 Vard\u00f8",
        "start": "1769-Jun-03 18:00",
        "stop": "1769-Jun-04 03:30",
        "step": "1m",
        "site_a": {
            "key": "TAHITI_POINT_VENUS",
            "label": "Tahiti Point Venus",
            "lon_deg_east": -149.4970,
            "lat_deg": -17.4950,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "VARDO_NORWAY",
            "label": "Vard\u00f8 Norway",
            "lon_deg_east": 31.1107,
            "lat_deg": 70.3706,
            "height_m": 0.0,
        },
    },
    {
        "title": "Cape Town \u2192 Reykjavík",
        "start": "2012-Jun-05 20:00",
        "stop": "2012-Jun-06 07:30",
        "step": "1m",
        "site_a": {
            "key": "CAPE_TOWN",
            "label": "Cape Town",
            "lon_deg_east": 18.4241,
            "lat_deg": -33.9249,
            "height_m": 0.0,
        },
        "site_b": {
            "key": "REYKJAVIK",
            "label": "Reykjavík",
            "lon_deg_east": -21.9426,
            "lat_deg": 64.1466,
            "height_m": 0.0,
        },
    },
]

try:
    import numpy as np
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "numpy", "pandas"])
    import numpy as np
    import pandas as pd

try:
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "scipy"])
    from scipy.interpolate import CubicSpline
    from scipy.optimize import minimize_scalar, brentq

try:
    from astroquery.jplhorizons import Horizons
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astroquery"])
    from astroquery.jplhorizons import Horizons

try:
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "astropy"])
    import astropy.units as u
    from astropy.time import Time
    from astropy.coordinates import EarthLocation
    from astropy.utils import iers

# Use bundled/fetched IERS tables when available; allow degraded mode in Colab if network table access is unavailable.
iers.conf.auto_download = True
iers.conf.iers_degraded_accuracy = "warn"


def norm(v):
    return float(np.sqrt(np.dot(v, v)))


def unit(v):
    n = norm(v)
    if n == 0.0:
        raise RuntimeError("Zero vector.")
    return np.asarray(v, dtype=float) / n


def angular_sep_arcsec(a, b):
    c = float(np.clip(np.dot(unit(a), unit(b)), -1.0, 1.0))
    return math.acos(c) * ARCSEC_PER_RAD


def horizons_geocenter_vectors(target_id, prefix):
    obj = Horizons(
        id=target_id,
        location="500@399",
        epochs={"start": START, "stop": STOP, "step": STEP},
    )
    vec = obj.vectors().to_pandas()
    out = pd.DataFrame()
    out["jd_tdb"] = vec["datetime_jd"].astype(float)
    out["utc"] = vec["datetime_str"].astype(str)
    out[f"{prefix}_x_km"] = vec["x"].astype(float) * AU_KM
    out[f"{prefix}_y_km"] = vec["y"].astype(float) * AU_KM
    out[f"{prefix}_z_km"] = vec["z"].astype(float) * AU_KM
    return out


def build_master_geocenter():
    sun = horizons_geocenter_vectors("10", "GEOCENTER_SUN")
    venus = horizons_geocenter_vectors("299", "GEOCENTER_VENUS")
    return sun.merge(venus, on=["jd_tdb", "utc"], how="inner")


def build_cache(df):
    cache = {"jd_tdb": df["jd_tdb"].to_numpy(dtype=float), "utc": df["utc"].astype(str).tolist()}
    for col in df.columns:
        if col.endswith("_km"):
            cache[col] = CubicSpline(cache["jd_tdb"], df[col].to_numpy(dtype=float), bc_type="natural")
    return cache


def vec_at(cache, prefix, jd_tdb):
    return np.array([
        float(cache[f"{prefix}_x_km"](jd_tdb)),
        float(cache[f"{prefix}_y_km"](jd_tdb)),
        float(cache[f"{prefix}_z_km"](jd_tdb)),
    ], dtype=float)


def utc_at(cache, jd_tdb):
    try:
        return Time(jd_tdb, format="jd", scale="tdb").utc.iso.replace(" ", " ")
    except Exception:
        jds = cache["jd_tdb"]
        idx = int(np.argmin(np.abs(jds - jd_tdb)))
        return cache["utc"][idx]


def earth_location(site):
    # Astropy EarthLocation uses the ERFA/SOFA geodetic-to-geocentric implementation.
    # Geodetic longitude is east-positive, matching Horizons convention used in this project.
    return EarthLocation.from_geodetic(
        lon=site["lon_deg_east"] * u.deg,
        lat=site["lat_deg"] * u.deg,
        height=site["height_m"] * u.m,
    )


def observer_gcrs_km(site, jd_tdb):
    # IERS path: geodetic site -> ITRS -> GCRS, implemented through Astropy/ERFA using IERS EOP tables.
    # The time is supplied as TDB and internally converted by Astropy as required for Earth orientation.
    t = Time(jd_tdb, format="jd", scale="tdb")
    loc = earth_location(site)
    pos, _vel = loc.get_gcrs_posvel(t)
    return np.array([pos.x.to_value(u.km), pos.y.to_value(u.km), pos.z.to_value(u.km)], dtype=float)


def topocentric_body_vector(cache, site, body_prefix, jd_tdb):
    return vec_at(cache, body_prefix, jd_tdb) - observer_gcrs_km(site, jd_tdb)


def site_sep_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    return angular_sep_arcsec(sun, venus)


def geocenter_sep_arcsec(cache, jd_tdb):
    return angular_sep_arcsec(vec_at(cache, "GEOCENTER_SUN", jd_tdb), vec_at(cache, "GEOCENTER_VENUS", jd_tdb))


def angular_radii_arcsec(cache, site, jd_tdb):
    sun = topocentric_body_vector(cache, site, "GEOCENTER_SUN", jd_tdb)
    venus = topocentric_body_vector(cache, site, "GEOCENTER_VENUS", jd_tdb)
    rs = math.atan2(SUN_RADIUS_KM, norm(sun)) * ARCSEC_PER_RAD
    rv = math.atan2(VENUS_RADIUS_KM, norm(venus)) * ARCSEC_PER_RAD
    return rs, rv


def contact_function(cache, site, event, jd_tdb):
    sep = site_sep_arcsec(cache, site, jd_tdb)
    rs, rv = angular_radii_arcsec(cache, site, jd_tdb)
    threshold = rs + rv if event in ["C1", "C4"] else rs - rv
    return sep - threshold


def find_roots(cache, site, event):
    jds = cache["jd_tdb"]
    vals = np.array([contact_function(cache, site, event, jd) for jd in jds], dtype=float)
    roots = []
    for i in range(len(jds) - 1):
        if not np.isfinite(vals[i]) or not np.isfinite(vals[i + 1]):
            continue
        if vals[i] == 0.0:
            roots.append(float(jds[i]))
        elif vals[i] * vals[i + 1] < 0.0:
            roots.append(float(brentq(
                lambda x: contact_function(cache, site, event, x),
                jds[i],
                jds[i + 1],
                xtol=1e-13,
                rtol=1e-13,
                maxiter=100,
            )))
    return roots


def contact_range(cache):
    roots = []
    for site in [SITE_A, SITE_B]:
        roots.extend(find_roots(cache, site, "C1"))
        roots.extend(find_roots(cache, site, "C4"))
    if not roots:
        raise RuntimeError("No C1/C4 contacts found.")
    return min(roots), max(roots)


def find_closest(cache):
    jds = cache["jd_tdb"]
    vals = [geocenter_sep_arcsec(cache, jd) for jd in jds]
    i = int(np.argmin(vals))
    a = jds[max(0, i - 3)]
    b = jds[min(len(jds) - 1, i + 3)]
    res = minimize_scalar(
        lambda jd: geocenter_sep_arcsec(cache, jd),
        bounds=(a, b),
        method="bounded",
        options={"xatol": 1e-13},
    )
    return float(res.x)


def fixed_geocenter_basis(cache, jd_tdb):
    sun = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    n = unit(sun)
    ref = np.array([0.0, 0.0, 1.0])
    if norm(np.cross(ref, n)) < 1e-12:
        ref = np.array([1.0, 0.0, 0.0])
    xhat = unit(np.cross(ref, n))
    yhat = unit(np.cross(n, xhat))
    return n, xhat, yhat


def ray_screen_point_arcsec(cache, site, jd_tdb, basis):
    n, xhat, yhat = basis
    obs = observer_gcrs_km(site, jd_tdb)
    sun_geo = vec_at(cache, "GEOCENTER_SUN", jd_tdb)
    venus_geo = vec_at(cache, "GEOCENTER_VENUS", jd_tdb)
    ray = venus_geo - obs
    screen_point = sun_geo
    denom = float(np.dot(ray, n))
    if abs(denom) < 1e-14:
        raise RuntimeError("Ray nearly parallel to solar screen.")
    tau = float(np.dot(screen_point - obs, n) / denom)
    hit = obs + tau * ray
    screen_vec = hit - sun_geo
    es = norm(sun_geo)
    x = math.atan2(float(np.dot(screen_vec, xhat)), es) * ARCSEC_PER_RAD
    y = math.atan2(float(np.dot(screen_vec, yhat)), es) * ARCSEC_PER_RAD
    return np.array([x, y], dtype=float)


def pca_direction(points):
    pts = np.asarray(points, dtype=float)
    mu = pts.mean(axis=0)
    centered = pts - mu
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    d = vt[0]
    if d[0] < 0:
        d = -d
    return mu, unit(d)


def line_intersection(mu, d, mid, normal):
    a = np.column_stack([d, -normal])
    b = mid - mu
    sol, *_ = np.linalg.lstsq(a, b, rcond=None)
    return mu + sol[0] * d


def fit_r2(points, degree):
    pts = np.asarray(points, dtype=float)
    x = pts[:, 0] - np.mean(pts[:, 0])
    y = pts[:, 1]
    coeff = np.polyfit(x, y, degree)
    yhat = np.polyval(coeff, x)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")


def distances_at(cache, jd_tdb):
    es = norm(vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    ev = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb))
    vs = norm(vec_at(cache, "GEOCENTER_VENUS", jd_tdb) - vec_at(cache, "GEOCENTER_SUN", jd_tdb))
    return es, ev, vs


def compute_iers_guideline_method(cache):
    ca_jd = find_closest(cache)
    c1, c4 = contact_range(cache)
    use_jds = cache["jd_tdb"][(cache["jd_tdb"] >= c1) & (cache["jd_tdb"] <= c4)]
    basis = fixed_geocenter_basis(cache, ca_jd)

    points = {}
    mus = {}
    dirs = {}
    track_rows = []

    for site in [SITE_A, SITE_B]:
        pts = np.array([ray_screen_point_arcsec(cache, site, jd, basis) for jd in use_jds], dtype=float)
        mu, d = pca_direction(pts)
        points[site["key"]] = pts
        mus[site["key"]] = mu
        dirs[site["key"]] = d
        angle = math.degrees(math.atan2(d[1], d[0]))
        track_rows.append({
            "track": site["label"],
            "lat": site["lat_deg"],
            "lon": site["lon_deg_east"],
            "angle_deg": angle,
            "linear_R2": fit_r2(pts, 1),
            "quad_R2": fit_r2(pts, 2),
            "cubic_R2": fit_r2(pts, 3),
            "parabolic_R2": fit_r2(pts, 2),
        })

    tangent = unit(dirs[SITE_A["key"]] + dirs[SITE_B["key"]])
    if tangent[0] < 0:
        tangent = -tangent
    normal = np.array([-tangent[1], tangent[0]])
    common_angle = math.degrees(math.atan2(tangent[1], tangent[0]))

    mid = 0.5 * (mus[SITE_A["key"]] + mus[SITE_B["key"]])
    aprime = line_intersection(mus[SITE_A["key"]], dirs[SITE_A["key"]], mid, normal)
    bprime = line_intersection(mus[SITE_B["key"]], dirs[SITE_B["key"]], mid, normal)

    abp_vec = bprime - aprime
    abp_arcsec = float(np.sqrt(np.sum(abp_vec * abp_vec)))
    sep_arcsec = abs(float(np.dot(abp_vec, normal)))

    es, ev, vs = distances_at(cache, ca_jd)
    abp_km = math.tan(abp_arcsec / ARCSEC_PER_RAD) * es
    baseline_proj_km = abp_km * ev / vs
    raw_phi = sep_arcsec * (ev / vs) * (EARTH_RADIUS_KM / baseline_proj_km)
    iau_phi = raw_phi * (es / AU_KM)
    delta = iau_phi - IAU_SOLAR_PARALLAX_ARCSEC

    theta_mean = 0.5 * (track_rows[0]["angle_deg"] + track_rows[1]["angle_deg"])
    delta_percent = 100.0 * delta / IAU_SOLAR_PARALLAX_ARCSEC

    rho_R_earth_arcsec = sep_arcsec * EARTH_RADIUS_KM / baseline_proj_km

    parallax = {
        "baseline_proj_km": baseline_proj_km,
        "A_prime_B_prime_arcsec": abp_arcsec,
        "A_prime_B_prime_km": abp_km,
        "sep_arcsec": sep_arcsec,
        "rho_R_earth_arcsec": rho_R_earth_arcsec,
        "raw_phi_arcsec": raw_phi,
        "IAU_phi_arcsec": iau_phi,
        "IAU_delta_arcsec": delta,
        "IAU_delta_percent": delta_percent,
        "common_angle_deg": common_angle,
        "theta_mean_deg": theta_mean,
        "D_ES_AU": es / AU_KM,
        "D_EV_D_VS": ev / vs,
        "IAU_R_EARTH_KM": IAU_EARTH_EQUATORIAL_RADIUS_KM,
        "IAU_AU_KM": AU_KM,
        "IAU_PI_TARGET_ARCSEC": IAU_SOLAR_PARALLAX_ARCSEC,
        "CA_utc": utc_at(cache, ca_jd),
        "CA_jd_tdb": ca_jd,
    }
    return parallax, track_rows






def html_value(value, unit):
    if isinstance(value, (int, float, np.floating)) and np.isfinite(value):
        if unit == "km":
            return f"{value:,.6f}"
        if unit == "d":
            return f"{value:.12f}"
        if unit == "%":
            return f"{value:.6f}"
        return f"{value:.6f}"
    return str(value)


def site_short_label(label):
    name = str(label).strip()
    low = name.lower()
    if low.startswith("north"):
        return "NP"
    if low.startswith("miami"):
        return "MIA"
    if "tahiti" in low:
        return "TAHITI"
    if "vard" in low or "vård" in low or "vør" in low or "vø" in low:
        return "VARDO"
    if low.startswith("cape"):
        return "CAPE"
    if low.startswith("reyk"):
        return "REYK"
    return name.split()[0].upper()[:6]


def build_report_rows(parallax, track_rows):
    parallax_rows = [
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_arcsec"], "arcsec"),
        ("A′B′<sub>OBS</sub>", parallax["A_prime_B_prime_km"], "km"),
        ("ρ<sub>OBS</sub>", parallax["sep_arcsec"], "arcsec"),
        ("ρ<sub>R⊕</sub>", parallax["rho_R_earth_arcsec"], "arcsec"),
        ("B<sub>eff,OBS</sub>", parallax["baseline_proj_km"], "km"),
        ("π<sub>OBS</sub>", parallax["raw_phi_arcsec"], "arcsec"),
        ("π<sub>IAU</sub>", parallax["IAU_phi_arcsec"], "arcsec"),
        ("Δπ", parallax["IAU_delta_arcsec"], "arcsec"),
        ("Δπ<sub>%</sub>", parallax["IAU_delta_percent"], "%"),
    ]

    track_rows_out = []
    for tr in track_rows:
        short = site_short_label(tr["track"])
        track_rows_out.append((f"θ<sub>{short}</sub>", tr["angle_deg"], "deg", tr["linear_R2"], tr["quad_R2"], tr["cubic_R2"]))

    track_rows_out.append(("θ̄", parallax["theta_mean_deg"], "deg", "", "", ""))
    track_rows_out.append(("θ<sub>COMMON</sub>", parallax["common_angle_deg"], "deg", "", "", ""))

    time_rows = [
        ("UTC<sub>CA</sub>", parallax["CA_utc"], ""),
        ("JD<sub>TDB,CA</sub>", parallax["CA_jd_tdb"], "d"),
        ("D<sub>ES</sub>/AU", parallax["D_ES_AU"], ""),
        ("D<sub>EV</sub>/D<sub>VS</sub>", parallax["D_EV_D_VS"], ""),
        ("R<sub>⊕,IAU</sub>", parallax["IAU_R_EARTH_KM"], "km"),
        ("AU<sub>IAU</sub>", parallax["IAU_AU_KM"], "km"),
        ("π<sub>IAU,TARGET</sub>", parallax["IAU_PI_TARGET_ARCSEC"], "arcsec"),
    ]

    return parallax_rows, track_rows_out, time_rows


def html_table(title, headers, rows, col_widths):
    html = []
    html.append(f"<div class='iers-title'>{title}</div>")
    html.append("<table class='iers-table'>")
    html.append("<colgroup>")
    for w in col_widths:
        html.append(f"<col style='width:{w};'>")
    html.append("</colgroup>")
    html.append("<thead><tr>")
    for i, h in enumerate(headers):
        cls = "num" if i == 1 or i >= 3 else ""
        html.append(f"<th class='{cls}'>{h}</th>")
    html.append("</tr></thead><tbody>")
    for row in rows:
        html.append("<tr>")
        for i, cell in enumerate(row):
            cls = "num" if i == 1 or i >= 3 else ""
            html.append(f"<td class='{cls}'>{cell}</td>")
        html.append("</tr>")
    html.append("</tbody></table>")
    return "".join(html)


def display_html_report(parallax, track_rows, csv_path, pair_title):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)

    parallax_html_rows = [[q, html_value(v, u), u] for q, v, u in parallax_rows]

    track_html_rows = []
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        track_html_rows.append([
            q,
            html_value(theta, unit),
            unit,
            "" if lin == "" else f"{float(lin):.9f}",
            "" if quad == "" else f"{float(quad):.9f}",
            "" if cubic == "" else f"{float(cubic):.9f}",
        ])

    time_html_rows = [[q, html_value(v, u), u] for q, v, u in time_rows]

    css = """
    <style>
    .iers-wrap {
        background:#000000;
        color:#f2f2f2;
        font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, "Liberation Mono", monospace;
        padding: 12px 14px;
        width: 760px;
        border: 1px solid #666;
        margin: 12px 0 18px 0;
    }
    .iers-title {
        color:#ffffff;
        font-weight:700;
        letter-spacing:0.05em;
        text-align:center;
        border-top:1px solid #aaa;
        border-bottom:1px solid #aaa;
        padding:6px 0;
        margin:10px 0 8px 0;
    }
    .iers-table {
        width:100%;
        border-collapse:collapse;
        margin-bottom:10px;
        table-layout:fixed;
    }
    .iers-table th {
        color:#ffffff;
        font-weight:700;
        border-bottom:1px solid #aaa;
        padding:5px 6px;
        text-align:left;
        white-space:nowrap;
    }
    .iers-table th.num {
        text-align:right;
    }
    .iers-table td {
        color:#f2f2f2;
        padding:4px 6px;
        border-bottom:1px solid #333;
        white-space:nowrap;
        text-align:left;
    }
    .iers-table td.num {
        text-align:right;
        font-weight:700;
        font-variant-numeric: tabular-nums;
    }
    .iers-table sub {
        font-size:65%;
        line-height:0;
        vertical-align:sub;
    }
    .iers-foot {
        margin-top:10px;
        color:#ddd;
        font-size:12px;
        word-break:break-all;
    }
    </style>
    """

    html = css
    html += "<div class='iers-wrap'>"
    html += html_table(
        f"IERS PARALLAX SOLUTION — {pair_title}",
        ["Quantity", "Value", "Units"],
        parallax_html_rows,
        ["34%", "42%", "24%"],
    )
    html += html_table(
        "TRACK GEOMETRY",
        ["Quantity", "Value", "Units", "R² lin", "R² quad", "R² cubic"],
        track_html_rows,
        ["18%", "18%", "12%", "17%", "17%", "18%"],
    )
    html += html_table(
        "REFERENCE GEOMETRY",
        ["Quantity", "Value", "Units"],
        time_html_rows,
        ["34%", "52%", "14%"],
    )
    html += f"<div class='iers-foot'>CSV: {csv_path}</div>"
    html += "</div>"

    try:
        from IPython.display import HTML, display
        display(HTML(html))
        return True
    except Exception:
        return False


def plain_label(label):
    return (
        str(label)
        .replace("<sub>", "₍")
        .replace("</sub>", "₎")
    )


def print_rule(width=74):
    print("─" * width)


def print_plain_section(title, rows, width=74):
    print()
    print("═" * width)
    print(title.center(width))
    print("═" * width)
    print(f"{'Quantity':<22} {'Value':>22} {'Units':<10}")
    print_rule(width)
    for q, v, u in rows:
        print(f"{plain_label(q):<22} {html_value(v, u):>22} {u:<10}")
    print_rule(width)


def print_plain_track_section(track_rows_out, width=74):
    print()
    print("═" * width)
    print("TRACK GEOMETRY".center(width))
    print("═" * width)
    print(f"{'Quantity':<18} {'Value':>12} {'Units':<8} {'R² lin':>12} {'R² cubic':>12}")
    print_rule(width)
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        lin_s = "" if lin == "" else f"{float(lin):.9f}"
        cubic_s = "" if cubic == "" else f"{float(cubic):.9f}"
        print(f"{plain_label(q):<18} {html_value(theta, unit):>12} {unit:<8} {lin_s:>12} {cubic_s:>12}")
    print_rule(width)


def write_csv(parallax, track_rows, out_csv):
    parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
    rows = []
    for q, v, u in parallax_rows:
        rows.append({"Section": "PARALLAX", "Quantity": plain_label(q), "Value": v, "Unit": u})
    for q, theta, unit, lin, quad, cubic in track_rows_out:
        rows.append({"Section": "TRACK", "Quantity": plain_label(q), "Value": theta, "Unit": unit})
        if lin != "":
            rows.append({"Section": "TRACK", "Quantity": plain_label(q) + " R2_LINEAR", "Value": lin, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": plain_label(q) + " R2_QUADRATIC", "Value": quad, "Unit": ""})
            rows.append({"Section": "TRACK", "Quantity": plain_label(q) + " R2_CUBIC", "Value": cubic, "Unit": ""})
    for q, v, u in time_rows:
        rows.append({"Section": "REFERENCE", "Quantity": plain_label(q), "Value": v, "Unit": u})
    pd.DataFrame(rows).to_csv(out_csv, index=False, float_format="%.12f")


def run_pair(pair, out_dir):
    global START, STOP, STEP, SITE_A, SITE_B

    START = pair["start"]
    STOP = pair["stop"]
    STEP = pair["step"]
    SITE_A = pair["site_a"]
    SITE_B = pair["site_b"]

    df = build_master_geocenter()
    cache = build_cache(df)
    parallax, track_rows = compute_iers_guideline_method(cache)

    safe_title = (
        pair["title"]
        .replace(" ", "_")
        .replace("→", "TO")
        .replace("ø", "o")
        .replace("Ø", "O")
        .replace("í", "i")
        .replace("Í", "I")
        .replace("/", "_")
    )

    out_csv = os.path.join(
        out_dir,
        f"{VERSION}_{safe_title}_IAU_CONSTANT_AUDIT_REPORT.csv",
    )
    write_csv(parallax, track_rows, out_csv)

    rendered = display_html_report(parallax, track_rows, out_csv, pair["title"])

    if not rendered:
        parallax_rows, track_rows_out, time_rows = build_report_rows(parallax, track_rows)
        print_plain_section(f"IERS PARALLAX SOLUTION — {pair['title']}", parallax_rows)
        print_plain_track_section(track_rows_out)
        print_plain_section("REFERENCE GEOMETRY", time_rows)
        print(f"CSV: {out_csv}")

def main():
    print(f"CODE OUTPUT: {VERSION}")

    out_dir = "/content/IERS_TN36_V01_MASTER_OUTPUT"
    os.makedirs(out_dir, exist_ok=True)

    for pair in PAIR_RUNS:
        run_pair(pair, out_dir)

    print(datetime.now(LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S %z"))
    print(f"END OF CODE OUTPUT: {VERSION}")


if __name__ == "__main__":
    main()
# IERS-0009


Saving IERS_0009_TN36_IAU_CONSTANT_AUDIT.py to IERS_0009_TN36_IAU_CONSTANT_AUDIT.py
CODE OUTPUT: IERS-0009


2026-07-08 20:00:33 -0500
END OF CODE OUTPUT: IERS-0009
